In [ ]:
import numpy as np
import pandas as pd
from timeit import default_timer
from psycopg2 import connect
from datetime import datetime, timedelta
import warnings
import os


In [ ]:
# Raw telematics data
raw_df = pd.read_csv(
    os.path.join("Outputs", "raw_df.csv"),
    encoding="utf-8"
)

raw_df

## Clean and check the data

### Min / max sanity checks

In [ ]:
df = raw_df.copy()

summary_df = pd.DataFrame({
    "column": ["event_ts", "odometer", "terminal_event_id"],
    "min": [
        pd.to_datetime(df["event_ts"], errors="coerce").min(),
        pd.to_numeric(df["odometer"], errors="coerce").min(),
        df["terminal_event_id"].min()
    ],
    "max": [
        pd.to_datetime(df["event_ts"], errors="coerce").max(),
        pd.to_numeric(df["odometer"], errors="coerce").max(),
        df["terminal_event_id"].max()
    ]
})

display(summary_df)


### Per-vehicle odometer range

In [ ]:
df = raw_df.copy()
df["odometer_num"] = pd.to_numeric(df["odometer"], errors="coerce")

odo_range = (
    df.groupby("vehicle_id")["odometer_num"]
      .agg(min_odo="min", max_odo="max")
      .reset_index()
)

odo_range["odo_range"] = odo_range["max_odo"] - odo_range["min_odo"]

display(odo_range.sort_values("odo_range", ascending=False))

### Event frequency by vehicle

- Identifies vehicles producing abnormal event volumes

In [ ]:
df = raw_df.copy()

event_rate = (
    df.assign(event_description=df["event_description"].astype("string").str.strip())
      .groupby(["vehicle_id", "event_description"])
      .size()
      .rename("event_count")
      .reset_index()
)

# Total events per vehicle
vehicle_totals = (
    df.groupby("vehicle_id")
      .size()
      .rename("total_event_count")
      .reset_index()
)

event_rate = event_rate.merge(vehicle_totals, on="vehicle_id", how="left")
event_rate["event_pct_of_vehicle_total"] = (event_rate["event_count"] / event_rate["total_event_count"]) * 100

# Top 3 events per vehicle
event_rate["rank"] = (
    event_rate.groupby("vehicle_id")["event_count"]
    .rank(method="first", ascending=False)
)

event_rate_top3 = (
    event_rate.loc[event_rate["rank"] <= 3]
    .sort_values(["vehicle_id", "rank"])
    .drop(columns="rank")
    .reset_index(drop=True)
)

display(event_rate_top3)


### Terminal_event_id uniqueness - To make sure no duplicate events. This is important as no terminal event ids should ever be the same. Each is unique as they are used as foreign keys to label each event, for each vehicle. 

In [ ]:
df = raw_df.copy()

# Flag duplicates
df["flag_dup_terminal_event_id"] = df["terminal_event_id"].duplicated(keep=False)

# Inspect
duplicates_df = (
    df.loc[df["flag_dup_terminal_event_id"], ["terminal_event_id", "vehicle_id", "event_ts", "event_description", "odometer"]]
    .sort_values(["terminal_event_id", "vehicle_id", "event_ts"])
    .reset_index(drop=True)
)

print("Total rows:", len(df))
print("Unique terminal_event_id:", df["terminal_event_id"].nunique(dropna=False))
print("Duplicate rows:", int(df["flag_dup_terminal_event_id"].sum()))
print("\nSample duplicates:")
print(duplicates_df.head(20))


### Time sanity: parse event_ts to timezone-aware

Must parse to timezone-aware timestamps

You cannot do any sequencing, trip segmentation, time-deltas, or non-decreasing timestamp checks safely unless event_ts is a real datetime type.

pd.to_datetime(..., utc=True) gives you:
- Consistent timezone-aware timestamps (UTC) so comparisons are valid.
- A flag for bad rows (NaT) where timestamps are malformed or missing.
- A clean base for later steps where you normalise by time if odometer is missing.

In [ ]:
df = raw_df.copy()

df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)
df["flag_event_ts_unparseable"] = df["event_ts_parsed"].isna()

total = len(df)
bad = int(df["flag_event_ts_unparseable"].sum())
pct_bad = (bad / total * 100) if total else 0.0

print(f"Unparseable timestamps: {bad}/{total} ({pct_bad:.2f}%)")

print("\nSample unparseable rows:")
print(
    df.loc[df["flag_event_ts_unparseable"], ["terminal_event_id", "vehicle_id", "event_ts", "event_description"]]
    .head(20)
    .reset_index(drop=True)
)


### Time sanity: per vehicle_id must be non-decreasing (with tolerance of 0.1 seconds)

It checks that within each vehicle_id, events are ordered by time. If you ever see an event timestamp that is earlier than the previous event for that same vehicle, that indicates one of:
- ingestion arriving out of order
- clock problems
- wrong vehicle mapping
- wrong timestamp parsing

We allow a small tolerance (0.1 seconds) because async ingestion can flip two adjacent events.

In [ ]:
df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)

tolerance = pd.Timedelta("0.1s")

mask = df["event_ts_parsed"].notna() & df["vehicle_id"].notna()
d = df.loc[mask].copy()

d = d.sort_values(["vehicle_id", "event_ts_parsed", "terminal_event_id"])
d["prev_ts"] = d.groupby("vehicle_id")["event_ts_parsed"].shift(1)
d["dt"] = d["event_ts_parsed"] - d["prev_ts"]

# flagged only when it is meaningfully earlier than prev_ts (beyond tolerance)
d["flag_time_out_of_order"] = d["dt"] < -tolerance

viol = d.loc[d["flag_time_out_of_order"]].copy()
viol["seconds_out_of_order"] = viol["dt"].dt.total_seconds()

print("Out-of-order rows:", len(viol))
print("\nSample out-of-order rows (shows how far backwards time went):")
print(
    viol[["terminal_event_id", "vehicle_id", "event_description", "prev_ts", "event_ts_parsed", "seconds_out_of_order"]]
    .head(20)
    .reset_index(drop=True)
)


### Vehicle consistency: “single vehicle_id per trip sequence” (Ign ON/OFF based)

Why: You need trip segments to compute per-trip features (counts, durations, risk events per trip). A common first segmentation is ignition:
- Trip starts at Ign ON
- Trip ends at Ign OFF

This also helps verify your data is coherent for “trip-based” analysis.

With 32% of inferred trips lasting under two minutes, the resulting trip distribution is highly unlikely for normal passenger vehicles. At the same time, the presence of a substantial proportion of longer trips (62% exceeding five minutes and 4% exceeding one hour) indicates that genuine extended driving does occur. This combination strongly suggests false trip splitting, where continuous journeys are fragmented into multiple short trips.

As a result, ignition-based trip segmentation (Ign ON–Ign OFF) does not reliably represent true trip boundaries in this dataset, likely due to stop start systems or device-level ignition inference. Consequently, trip identification based solely on ignition events was deemed unreliable. Alternative segmentation strategies, such as time-gap-based trips, are required. Where trip reconstruction remains ambiguous, aggregation at the daily level provides a conservative fallback to maintain consistency and analytical validity.

In [ ]:
df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)

# Sort per vehicle
df = df.sort_values(["vehicle_id", "event_ts_parsed", "terminal_event_id"], na_position="last").copy()

is_ign_on = df["event_description"].astype("string").str.strip().eq("Ign ON")
df["trip_counter"] = is_ign_on.groupby(df["vehicle_id"]).cumsum()
df["trip_id"] = df["vehicle_id"].astype("string") + "_trip_" + df["trip_counter"].astype("int64").astype("string")

# Check: each trip_id maps to exactly one vehicle_id (should be true by construction)
trip_vehicle_counts = (
    df.groupby("trip_id")["vehicle_id"]
    .nunique(dropna=False)
    .rename("vehicle_ids_in_trip")
    .reset_index()
)

inconsistent_trips_df = trip_vehicle_counts.loc[trip_vehicle_counts["vehicle_ids_in_trip"] != 1].copy()

print("Inconsistent trip_id count:", len(inconsistent_trips_df))
print(inconsistent_trips_df.head(20))



df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)

# Keep only rows with usable time
d = df.loc[df["event_ts_parsed"].notna() & df["vehicle_id"].notna()].copy()
d = d.sort_values(["vehicle_id", "event_ts_parsed", "terminal_event_id"])

# Define trip start
is_ign_on = d["event_description"].astype("string").str.strip().eq("Ign ON")

# Trip number increments at each Ign ON per vehicle
d["trip_no"] = is_ign_on.groupby(d["vehicle_id"]).cumsum().astype("int64")

# Trip id
d["trip_id"] = d["vehicle_id"].astype("string") + "_trip_" + d["trip_no"].astype("string")

# Trip summary
trip_summary = (
    d.groupby(["vehicle_id", "trip_id"], as_index=False)
     .agg(
         trip_start=("event_ts_parsed", "min"),
         trip_end=("event_ts_parsed", "max"),
         n_events=("terminal_event_id", "count"),
         n_ign_on=("event_description", lambda s: (s.astype("string").str.strip() == "Ign ON").sum()),
         n_ign_off=("event_description", lambda s: (s.astype("string").str.strip() == "Ign OFF").sum()),
     )
)

trip_summary["duration_seconds"] = (trip_summary["trip_end"] - trip_summary["trip_start"]).dt.total_seconds()

print("Trip summary (top 10):")
display(trip_summary.head(10).reset_index(drop=True))

# Show a worked example: pick the first trip that has an Ign ON (trip_no >= 1)
example = trip_summary.loc[trip_summary["trip_id"].notna()].head(1)
if len(example) == 0:
    print("\nNo trips found (no Ign ON events or no parseable timestamps).")
else:
    ex_trip_id = example.iloc[0]["trip_id"]
    print(f"\nExample trip_id: {ex_trip_id}")
    display(example.reset_index(drop=True))

    print("\nEvents in example trip:")
    display(
        d.loc[d["trip_id"] == ex_trip_id, ["terminal_event_id", "event_ts_parsed", "event_description", "odometer"]]
        .reset_index(drop=True)
    )


#### Looking at trip durations

In [ ]:
df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)

# Keep only rows we can sequence
d = df.loc[df["event_ts_parsed"].notna() & df["vehicle_id"].notna()].copy()
d = d.sort_values(["vehicle_id", "event_ts_parsed", "terminal_event_id"])

# Trip segmentation: starts at each Ign ON
d["is_ign_on"] = d["event_description"].astype("string").str.strip().eq("Ign ON")
d["trip_no"] = d["is_ign_on"].groupby(d["vehicle_id"]).cumsum().astype("int64")
d["trip_id"] = d["vehicle_id"].astype("string") + "_trip_" + d["trip_no"].astype("string")

# Trip summary with duration
trip_summary = (
    d.groupby(["vehicle_id", "trip_id"], as_index=False)
     .agg(
         trip_start=("event_ts_parsed", "min"),
         trip_end=("event_ts_parsed", "max"),
         n_events=("terminal_event_id", "count"),
         n_ign_on=("is_ign_on", "sum"),
         n_ign_off=("event_description", lambda s: (s.astype("string").str.strip() == "Ign OFF").sum()),
     )
)
trip_summary["duration_seconds"] = (trip_summary["trip_end"] - trip_summary["trip_start"]).dt.total_seconds()

# Helper to print % + examples
def report_bucket(trip_summary_df, label, mask, n_trips_total, d_events, n_examples=3, n_rows_per_example=30):
    bucket = trip_summary_df.loc[mask].copy()
    n_bucket = len(bucket)
    pct = (n_bucket / n_trips_total * 100) if n_trips_total else 0.0

    print(f"\n{label}")
    print(f"Trips: {n_bucket}/{n_trips_total} ({pct:.2f}%)")

    if n_bucket == 0:
        print("No examples.")
        return

    # Choose example trips: smallest, median-ish, largest duration within bucket
    bucket = bucket.sort_values("duration_seconds")
    pick_idx = [0, len(bucket) // 2, len(bucket) - 1]
    pick_idx = list(dict.fromkeys([i for i in pick_idx if 0 <= i < len(bucket)]))[:n_examples]
    examples = bucket.iloc[pick_idx].copy()

    for i, row in examples.reset_index(drop=True).iterrows():
        trip_id = row["trip_id"]
        vehicle_id = row["vehicle_id"]
        dur = row["duration_seconds"]
        start = row["trip_start"]
        end = row["trip_end"]
        n_events = row["n_events"]
        n_on = row["n_ign_on"]
        n_off = row["n_ign_off"]

        print(f"\nExample {i+1}: vehicle_id={vehicle_id} trip_id={trip_id}")
        print(f"Duration(s)={dur:.0f} | start={start} | end={end} | events={n_events} | IgnON={n_on} | IgnOFF={n_off}")

        ex_events = (
            d_events.loc[d_events["trip_id"] == trip_id, ["terminal_event_id", "event_ts_parsed", "event_description", "odometer"]]
            .sort_values(["event_ts_parsed", "terminal_event_id"])
            .head(n_rows_per_example)
            .reset_index(drop=True)
        )
        display(ex_events)

# Total trips
# Note: trip_no will be 0 before first Ign ON (if any). Those are "pre-trip" fragments.
# We only consider trip_no >= 1 as actual trips started by Ign ON.
trip_summary_real = trip_summary.loc[trip_summary["trip_id"].notna()].copy()
trip_summary_real = trip_summary_real.loc[trip_summary_real["trip_id"].str.contains("_trip_")].copy()

# filter out trip_no == 0 explicitly
trip_no_extracted = trip_summary_real["trip_id"].str.rsplit("_trip_", n=1, expand=True)[1]
trip_summary_real["trip_no"] = pd.to_numeric(trip_no_extracted, errors="coerce")
trip_summary_real = trip_summary_real.loc[trip_summary_real["trip_no"].fillna(0).astype(int) >= 1].copy()

n_trips_total = len(trip_summary_real)
print("Total trips (Ign ON started):", n_trips_total)

# Buckets
under_120s = trip_summary_real["duration_seconds"] < 120
over_5min = trip_summary_real["duration_seconds"] > 300
over_1hr = trip_summary_real["duration_seconds"] > 3600

report_bucket(trip_summary_real, "1) Trips under 120 seconds", under_120s, n_trips_total, d, n_examples=3, n_rows_per_example=40)
report_bucket(trip_summary_real, "2) Trips over 5 minutes", over_5min, n_trips_total, d, n_examples=3, n_rows_per_example=40)
report_bucket(trip_summary_real, "3) Trips over 1 hour", over_1hr, n_trips_total, d, n_examples=3, n_rows_per_example=40)


In [ ]:
df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)

# Sort and segment trips by Ign ON
d = df.loc[df["event_ts_parsed"].notna() & df["vehicle_id"].notna()].copy()
d = d.sort_values(["vehicle_id", "event_ts_parsed", "terminal_event_id"])

d["is_ign_on"] = d["event_description"].astype("string").str.strip().eq("Ign ON")
d["trip_no"] = d["is_ign_on"].groupby(d["vehicle_id"]).cumsum()
d["trip_id"] = d["vehicle_id"].astype("string") + "_trip_" + d["trip_no"].astype("string")

# Trip durations
trip_summary = (
    d.groupby("trip_id", as_index=False)
     .agg(
         trip_start=("event_ts_parsed", "min"),
         trip_end=("event_ts_parsed", "max")
     )
)
trip_summary["duration_seconds"] = (
    trip_summary["trip_end"] - trip_summary["trip_start"]
).dt.total_seconds()

# Only real trips (trip_no >= 1)
trip_summary["trip_no"] = pd.to_numeric(
    trip_summary["trip_id"].str.rsplit("_trip_", n=1).str[1],
    errors="coerce"
).fillna(0).astype(int)

trip_summary = trip_summary.loc[trip_summary["trip_no"] >= 1]

total_trips = len(trip_summary)

def report(label, mask):
    n = int(mask.sum())
    pct = (n / total_trips * 100) if total_trips else 0.0
    print(f"{label}\nTrips: {n}/{total_trips} ({pct:.2f}%)\n")

report("1) Trips under 120 seconds", trip_summary["duration_seconds"] < 120)
report("2) Trips over 5 minutes", trip_summary["duration_seconds"] > 300)
report("3) Trips over 1 hour", trip_summary["duration_seconds"] > 3600)


### Odometer sanity: non-negative

If odometer is missing, you cannot compute distance-based features reliably. You then normalise by time (events per minute, risk per minute, etc.). So you need to quantify missingness and decide how much of your dataset is distance-usable.

With very little odometer missing, we can still do analysis. But may focus on total distance by driver per month. As per trip might be inconsistent but per month should stay accurate due to little missing.

In [ ]:
df = raw_df.copy()
df["odometer_num"] = pd.to_numeric(df["odometer"], errors="coerce")
df["flag_odo_missing"] = df["odometer_num"].isna()

total = len(df)
missing = int(df["flag_odo_missing"].sum())
pct_missing = (missing / total * 100) if total else 0.0

print(f"Odometer missing: {missing}/{total} ({pct_missing:.2f}%)")

# Optional: missing by vehicle
missing_by_vehicle = (
    df.groupby("vehicle_id", dropna=False)["flag_odo_missing"]
      .mean()
      .mul(100)
      .rename("pct_odo_missing")
      .reset_index()
      .sort_values("pct_odo_missing", ascending=False)
)

print("\nTop vehicles by % missing odometer:")
print(missing_by_vehicle.head(20).reset_index(drop=True))


### Odometer sanity: non-decreasing and no jumps within vehicle (with tolerance)

Why this test is necessary:
- Huge jumps often mean unit mismatch, odometer reset, device replacement, bad decoding, or wrong vehicle mapping.
- If you compute distance, these jumps can create nonsense like 200 km in 2 seconds.

In [ ]:
df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)
df["odometer_num"] = pd.to_numeric(df["odometer"], errors="coerce")

jump_threshold = 200_000.0  # tune for your units

mask = df["event_ts_parsed"].notna() & df["vehicle_id"].notna() & df["odometer_num"].notna()
d = df.loc[mask].copy()

d = d.sort_values(["vehicle_id", "event_ts_parsed", "terminal_event_id"])
d["prev_odo"] = d.groupby("vehicle_id")["odometer_num"].shift(1)
d["prev_ts"] = d.groupby("vehicle_id")["event_ts_parsed"].shift(1)

d["dodo"] = d["odometer_num"] - d["prev_odo"]
d["flag_odo_jump"] = d["dodo"] > float(jump_threshold)

jumps = d.loc[d["flag_odo_jump"]].copy()
jumps["seconds_since_prev"] = (jumps["event_ts_parsed"] - jumps["prev_ts"]).dt.total_seconds()

print("Odometer jump rows:", len(jumps))

print("\nSample jumps (with previous value and timestamp):")
print(
    jumps[[
        "terminal_event_id",
        "vehicle_id",
        "prev_ts",
        "event_ts_parsed",
        "prev_odo",
        "odometer_num",
        "dodo",
        "seconds_since_prev",
        "event_description"
    ]]
    .head(30)
    .reset_index(drop=True)
)


### Per day trips. Not per IGN

In [ ]:
import pandas as pd
import numpy as np

df = raw_df.copy()
df["event_ts_parsed"] = pd.to_datetime(df["event_ts"], errors="coerce", utc=True)
df["odometer_num"] = pd.to_numeric(df["odometer"], errors="coerce")

# Keep rows we can date-bucket
d = df.loc[df["event_ts_parsed"].notna() & df["vehicle_id"].notna()].copy()
d["date_utc"] = d["event_ts_parsed"].dt.date  # daily bucket (UTC). change to .dt.tz_convert(...) if you want local day

# ------------------------------------------------------------
# "Trips per day" using time-gap segmentation (not IGN)
# New trip whenever gap >= X minutes within the same vehicle-day.
# ------------------------------------------------------------
gap_minutes = 10
gap = pd.Timedelta(minutes=gap_minutes)

d = d.sort_values(["vehicle_id", "date_utc", "event_ts_parsed", "terminal_event_id"])

d["prev_ts"] = d.groupby(["vehicle_id", "date_utc"])["event_ts_parsed"].shift(1)
d["gap"] = d["event_ts_parsed"] - d["prev_ts"]
d["new_trip"] = d["prev_ts"].isna() | (d["gap"] >= gap)

d["trip_no_day"] = d.groupby(["vehicle_id", "date_utc"])["new_trip"].cumsum().astype("int64")
d["trip_id_day"] = (
    d["vehicle_id"].astype("string")
    + "_"
    + d["date_utc"].astype("string")
    + "_trip_"
    + d["trip_no_day"].astype("string")
)

# Trip-level summary within each day
trip_day_summary = (
    d.groupby(["vehicle_id", "date_utc", "trip_no_day"], as_index=False)
     .agg(
         trip_start=("event_ts_parsed", "min"),
         trip_end=("event_ts_parsed", "max"),
         n_events=("terminal_event_id", "count"),
         n_unique_event_types=("event_description", lambda s: s.astype("string").str.strip().nunique()),
         odo_start=("odometer_num", "min"),
         odo_end=("odometer_num", "max"),
     )
)

trip_day_summary["duration_minutes"] = (trip_day_summary["trip_end"] - trip_day_summary["trip_start"]).dt.total_seconds() / 60

# Distance only where odometer exists for the trip
trip_day_summary["has_odo"] = trip_day_summary["odo_start"].notna() & trip_day_summary["odo_end"].notna()
trip_day_summary["odo_delta"] = np.where(
    trip_day_summary["has_odo"],
    trip_day_summary["odo_end"] - trip_day_summary["odo_start"],
    np.nan
)

display(trip_day_summary.sort_values(["vehicle_id", "date_utc", "trip_no_day"]).head(50))

# ------------------------------------------------------------
# Daily "rating base" summary (vehicle-day)
# - trips_per_day (gap-based)
# - total duration
# - odometer delta if usable
# - event counts and top event types
# ------------------------------------------------------------
daily_summary = (
    trip_day_summary.groupby(["vehicle_id", "date_utc"], as_index=False)
    .agg(
        trips_per_day=("trip_no_day", "max"),
        total_events=("n_events", "sum"),
        total_duration_minutes=("duration_minutes", "sum"),
        trips_with_odo=("has_odo", "sum"),
        daily_odo_start=("odo_start", "min"),
        daily_odo_end=("odo_end", "max"),
    )
)

daily_summary["daily_odo_delta"] = np.where(
    daily_summary["daily_odo_start"].notna() & daily_summary["daily_odo_end"].notna(),
    daily_summary["daily_odo_end"] - daily_summary["daily_odo_start"],
    np.nan
)

# Top 3 event types per vehicle-day (for interpretability)
event_day_counts = (
    d.assign(event_description=d["event_description"].astype("string").str.strip())
     .groupby(["vehicle_id", "date_utc", "event_description"])
     .size()
     .rename("event_count")
     .reset_index()
)

event_day_counts["rank"] = (
    event_day_counts.groupby(["vehicle_id", "date_utc"])["event_count"]
    .rank(method="first", ascending=False)
)

top3_event_day = (
    event_day_counts.loc[event_day_counts["rank"] <= 3]
    .sort_values(["vehicle_id", "date_utc", "rank"])
    .drop(columns="rank")
    .reset_index(drop=True)
)

display(daily_summary.sort_values(["vehicle_id", "date_utc"]).head(50))
display(top3_event_day.head(50))

# ------------------------------------------------------------
# Quick view: distribution of trips_per_day (does this look sane?)
# ------------------------------------------------------------
print("Trips per day distribution:")
print(daily_summary["trips_per_day"].describe())

print("\n% of vehicle-days with usable daily odometer delta:")
pct_odo = (daily_summary["daily_odo_delta"].notna().mean() * 100) if len(daily_summary) else 0.0
print(f"{pct_odo:.2f}%")


## Classify Vehicles

In [ ]:
import numpy as np
import pandas as pd

# =========================
# INPUT
# =========================
# raw_df must exist with columns:
# terminal_event_id, event_description, vehicle_id, event_ts, odometer

# =========================
# CONFIG
# =========================
SESSION_GAP_SECONDS = (3 * 60 + 30)  # 3m30s

# Kept internally for exposure estimation only.
# CHANGED DUE TO MANAGER FEEDBACK:
# Night driving is no longer used in scoring or visible ranking.
NIGHT_START_HOUR = 22
NIGHT_END_HOUR = 5

# CHANGED DUE TO MANAGER FEEDBACK:
# Use episode aggregation instead of raw total counts as the main fairness fix.
# Exposure is kept only for eligibility, not as a visible scored feature.

# Episode windows
HARSH_EPISODE_WINDOW_SECONDS = 5
SPEEDING_EPISODE_WINDOW_SECONDS = 180
POWER_EPISODE_WINDOW_SECONDS = 10 * 60
CAMERA_EPISODE_WINDOW_SECONDS = 60
FATIGUE_EPISODE_WINDOW_SECONDS = 30
DISTRACTION_EPISODE_WINDOW_SECONDS = 60
SITUATIONAL_EPISODE_WINDOW_SECONDS = 10

# CHANGED DUE TO MANAGER FEEDBACK:
# Vehicles with low exposure must not be ranked Low / Medium / High.
INSUFFICIENT_EXPOSURE_HOURS = 3.0

# =========================
# EVENT GROUPS
# =========================
# CHANGED DUE TO MANAGER FEEDBACK:
# Speeding removed from generic risky group and split into short/long speeding episodes.
HARSH_SUBSTRINGS = {
    "Corner": "harsh_corner_count",
    "Braking": "harsh_braking_count",
    "Accel": "harsh_accel_count",
}

SPEEDING_SUBSTRINGS = {
    "Speeding": "speeding_count",
}

# CHANGED DUE TO MANAGER FEEDBACK:
# Power only contains true power-related events.
POWER_SUBSTRINGS = {
    "Power OFF": "power_off_ext_batt_disc_count",
}

# CHANGED DUE TO MANAGER FEEDBACK:
# Camera covered split out into its own category.
CAMERA_SUBSTRINGS = {
    "v_cam_covered": "cam_covered_count",
}

# CHANGED DUE TO MANAGER FEEDBACK:
# AI-based attention split into meaningful sub-categories.
FATIGUE_SUBSTRINGS = {
    "v_eye_closed": "eye_closed_count",
    "v_yawn": "yawn_count",
    "v_fatigue": "fatigue_count",
}

DISTRACTION_SUBSTRINGS = {
    "v_distraction": "distraction_count",
    "v_phone": "phone_use_count",
    "v_smoke": "smoke_count",
    "SEATBELT_D_OFF": "Driver_Seatbelt_not_on",
}

SITUATIONAL_SUBSTRINGS = {
    "v_Headway_Mon": "headway_monitor_count",
    "v_lane_departure": "lane_departure_count",
    "v_fwd_collision": "forward_collision_count",
    "v_ped_collision": "ped_collision_count",
}

CRASH_SUBSTRINGS = {
    "CRASH": "crash_count",
}

# For detailed daily raw event counts appendix
ALL_EVENT_COUNT_MAPS = (
    list(HARSH_SUBSTRINGS.items())
    + list(SPEEDING_SUBSTRINGS.items())
    + list(POWER_SUBSTRINGS.items())
    + list(CAMERA_SUBSTRINGS.items())
    + list(FATIGUE_SUBSTRINGS.items())
    + list(DISTRACTION_SUBSTRINGS.items())
    + list(SITUATIONAL_SUBSTRINGS.items())
    + list(CRASH_SUBSTRINGS.items())
)

# =========================
# HELPERS
# =========================
def _map_event_to_proxy_bucket(event_desc: str) -> tuple[str, str]:
    s = ((event_desc or "").lower())

    for k, out_col in HARSH_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("harsh_manoeuvre_proxy", out_col)

    for k, out_col in SPEEDING_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("speeding_proxy", out_col)

    for k, out_col in POWER_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("power_violation_proxy", out_col)

    for k, out_col in CAMERA_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("camera_obstruction_proxy", out_col)

    for k, out_col in FATIGUE_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("fatigue_proxy", out_col)

    for k, out_col in DISTRACTION_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("driver_distraction_safety_proxy", out_col)

    for k, out_col in SITUATIONAL_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("situational_forward_risk_proxy", out_col)

    for k, out_col in CRASH_SUBSTRINGS.items():
        if (k.lower() in s):
            return ("crash_marker_analysis_only", out_col)

    return ("other", "not_used")

def _day_start(ts: pd.Timestamp) -> pd.Timestamp:
    return (ts.floor("D"))

def _split_interval_by_day(
    t0: pd.Timestamp, t1: pd.Timestamp
) -> list[tuple[pd.Timestamp, pd.Timestamp, pd.Timestamp]]:
    if (t1 <= t0):
        return []
    parts = []
    cur_start = (t0)
    while (cur_start < t1):
        d0 = (_day_start(cur_start))
        next_day = (d0 + pd.Timedelta(days=1))
        cur_end = (min(t1, next_day))
        parts.append((d0, cur_start, cur_end))
        cur_start = (cur_end)
    return parts

def _night_overlap_seconds(part_start: pd.Timestamp, part_end: pd.Timestamp) -> float:
    if (part_end <= part_start):
        return 0.0

    d0 = (_day_start(part_start))

    night1_start = (d0 + pd.Timedelta(hours=0))
    night1_end = (d0 + pd.Timedelta(hours=NIGHT_END_HOUR))

    night2_start = (d0 + pd.Timedelta(hours=NIGHT_START_HOUR))
    night2_end = (d0 + pd.Timedelta(days=1))

    def overlap(a0, a1, b0, b1):
        x0 = (max(a0, b0))
        x1 = (min(a1, b1))
        return (max(0.0, (x1 - x0).total_seconds()))

    return (
        overlap(part_start, part_end, night1_start, night1_end)
        + overlap(part_start, part_end, night2_start, night2_end)
    )

def _to_week_start(d: pd.Series) -> pd.Series:
    return ((d.dt.to_period("W-SUN").dt.start_time))

def _to_month_start(d: pd.Series) -> pd.Series:
    return (d.dt.to_period("M").dt.start_time)

def _rank01(s: pd.Series) -> pd.Series:
    if (s.nunique(dropna=True) <= 1):
        return (pd.Series(np.zeros(len(s)), index=s.index))
    return (s.rank(method="average", pct=True).fillna(0.0))

def _add_raw_counts(
    df: pd.DataFrame,
    desc_lower: pd.Series,
    counts_base: pd.DataFrame,
    substr_map: dict,
) -> pd.DataFrame:
    out = (counts_base.copy())
    for k, col in substr_map.items():
        mask = (desc_lower.str.contains(k.lower(), na=False))
        tmp = (
            df.loc[mask, ["vehicle_id", "day"]]
            .groupby(["vehicle_id", "day"], as_index=False)
            .size()
            .rename(columns={"size": col})
        )
        out = (out.merge(tmp, on=["vehicle_id", "day"], how="left"))
    return out

def _build_match_mask(desc_lower: pd.Series, substr_map: dict) -> pd.Series:
    mask = (pd.Series(False, index=desc_lower.index))
    for k in substr_map.keys():
        mask = (mask | desc_lower.str.contains(k.lower(), na=False))
    return mask

def _episode_counts_from_mask(
    df: pd.DataFrame,
    mask: pd.Series,
    gap_seconds: int,
    out_col: str,
) -> pd.DataFrame:
    subset = (
        df.loc[mask, ["vehicle_id", "event_ts"]]
        .sort_values(["vehicle_id", "event_ts"])
        .copy()
    )

    if (subset.empty):
        return pd.DataFrame(columns=["vehicle_id", "day", out_col])

    subset["prev_ts"] = (subset.groupby("vehicle_id")["event_ts"].shift(1))
    subset["gap_s"] = ((subset["event_ts"] - subset["prev_ts"]).dt.total_seconds())

    subset["new_episode"] = (
        subset["prev_ts"].isna()
        | (subset["gap_s"] > gap_seconds)
    ).astype("int64")

    subset["episode_id"] = (subset.groupby("vehicle_id")["new_episode"].cumsum())
    subset["day"] = (subset["event_ts"].dt.floor("D"))

    out = (
        subset.groupby(["vehicle_id", "day", "episode_id"], as_index=False)
        .size()
        .groupby(["vehicle_id", "day"], as_index=False)
        .size()
        .rename(columns={"size": out_col})
    )

    return out

def _speeding_episode_breakdown(df: pd.DataFrame, desc_lower: pd.Series) -> pd.DataFrame:
    # CHANGED DUE TO MANAGER FEEDBACK:
    # Speeding is now episode-based and split into short vs long speeding.
    mask = (_build_match_mask(desc_lower, SPEEDING_SUBSTRINGS))
    subset = (
        df.loc[mask, ["vehicle_id", "event_ts"]]
        .sort_values(["vehicle_id", "event_ts"])
        .copy()
    )

    if (subset.empty):
        return pd.DataFrame(
            columns=[
                "vehicle_id",
                "day",
                "short_speeding_episode_count",
                "long_speeding_episode_count",
                "speeding_episode_count",
            ]
        )

    subset["prev_ts"] = (subset.groupby("vehicle_id")["event_ts"].shift(1))
    subset["gap_s"] = ((subset["event_ts"] - subset["prev_ts"]).dt.total_seconds())
    subset["new_episode"] = (
        subset["prev_ts"].isna()
        | (subset["gap_s"] > SPEEDING_EPISODE_WINDOW_SECONDS)
    ).astype("int64")

    subset["episode_id"] = (subset.groupby("vehicle_id")["new_episode"].cumsum())
    subset["day"] = (subset["event_ts"].dt.floor("D"))

    episode_sizes = (
        subset.groupby(["vehicle_id", "day", "episode_id"], as_index=False)
        .size()
        .rename(columns={"size": "events_in_episode"})
    )

    episode_sizes["short_speeding_episode_count"] = (
        (episode_sizes["events_in_episode"] == 1).astype("int64")
    )
    episode_sizes["long_speeding_episode_count"] = (
        (episode_sizes["events_in_episode"] > 1).astype("int64")
    )
    episode_sizes["speeding_episode_count"] = 1

    out = (
        episode_sizes.groupby(["vehicle_id", "day"], as_index=False)[
            [
                "short_speeding_episode_count",
                "long_speeding_episode_count",
                "speeding_episode_count",
            ]
        ]
        .sum()
    )

    return out

# =========================
# AGGREGATION CONFIG
# =========================
agg_sum_cols = [
    "drive_seconds",
    "night_drive_seconds",
    "distance_km",
    "night_distance_km",
    "harsh_corner_count",
    "harsh_braking_count",
    "harsh_accel_count",
    "speeding_count",
    "power_off_ext_batt_disc_count",
    "cam_covered_count",
    "eye_closed_count",
    "yawn_count",
    "fatigue_count",
    "distraction_count",
    "phone_use_count",
    "smoke_count",
    "Driver_Seatbelt_not_on",
    "headway_monitor_count",
    "lane_departure_count",
    "forward_collision_count",
    "ped_collision_count",
    "crash_count",
    "harsh_episode_count",
    "short_speeding_episode_count",
    "long_speeding_episode_count",
    "speeding_episode_count",
    "power_violation_episode_count",
    "camera_obstruction_episode_count",
    "fatigue_episode_count",
    "driver_distraction_episode_count",
    "situational_risk_episode_count",
    "harsh_total_count",
    "fatigue_total_count",
    "driver_distraction_total_count",
    "situational_risk_total_count",
]

def _aggregate(features: pd.DataFrame, window: str) -> pd.DataFrame:
    x = (features.copy())

    if (window == "week"):
        x["window_start"] = (_to_week_start(x["day"]))
        x["window_end"] = (x["window_start"] + pd.Timedelta(days=7))
    elif (window == "month"):
        x["window_start"] = (_to_month_start(x["day"]))
        x["window_end"] = (x["window_start"] + pd.offsets.MonthBegin(1))
    else:
        raise ValueError("window must be 'week' or 'month'")

    out = (
        x.groupby(["vehicle_id", "window_start", "window_end"], as_index=False)
        .agg({c: "sum" for c in agg_sum_cols if (c in x.columns)})
    )

    out["drive_hours"] = (out["drive_seconds"] / 3600.0)
    out["night_ratio"] = (
        np.where(
            (out["drive_seconds"] > 0),
            (out["night_drive_seconds"] / out["drive_seconds"]),
            0.0,
        )
    )

    daily_known = (features.copy())
    daily_known["has_km"] = (daily_known["distance_km"].notna().astype("int64"))

    if (window == "week"):
        daily_known["window_start"] = (_to_week_start(daily_known["day"]))
    else:
        daily_known["window_start"] = (_to_month_start(daily_known["day"]))

    has_any_km = (
        daily_known.groupby(["vehicle_id", "window_start"], as_index=False)["has_km"]
        .max()
        .rename(columns={"has_km": "has_km_any"})
    )

    out = (out.merge(has_any_km, on=["vehicle_id", "window_start"], how="left"))
    out.loc[(out["has_km_any"] != 1), ["distance_km", "night_distance_km"]] = (np.nan)
    out = (out.drop(columns=["has_km_any"]))

    return (out.sort_values(["vehicle_id", "window_start"]).reset_index(drop=True))

def add_score_and_class(df_in: pd.DataFrame, time_col: str) -> pd.DataFrame:
    d = (df_in.copy())

    if ("window_end" not in d.columns):
        d["window_end"] = (d[time_col] + pd.Timedelta(days=1))

    # CHANGED DUE TO MANAGER FEEDBACK:
    # Drive hours and night ratio removed from visible scoring.
    # Exposure is only used to assign "Insufficient Exposure".
    d["r_harsh_total"] = (_rank01(d["harsh_episode_count"]))
    d["r_short_speeding"] = (_rank01(d["short_speeding_episode_count"]))
    d["r_long_speeding"] = (_rank01(d["long_speeding_episode_count"]))
    d["r_power_total"] = (_rank01(d["power_violation_episode_count"]))
    d["r_camera_total"] = (_rank01(d["camera_obstruction_episode_count"]))
    d["r_fatigue_total"] = (_rank01(d["fatigue_episode_count"]))
    d["r_distraction_total"] = (_rank01(d["driver_distraction_episode_count"]))
    d["r_situational_total"] = (_rank01(d["situational_risk_episode_count"]))

    # CHANGED DUE TO MANAGER FEEDBACK:
    # Power reduced, camera split out, harsh and speeding increased,
    # AI attention split into meaningful categories.
    w_harsh = 0.20
    w_short_speed = 0.12
    w_long_speed = 0.18
    w_power = 0.10
    w_camera = 0.05
    w_fatigue = 0.12
    w_distraction = 0.13
    w_situational = 0.10

    d["ubi_proxy_score"] = (
        (w_harsh * d["r_harsh_total"])
        + (w_short_speed * d["r_short_speeding"])
        + (w_long_speed * d["r_long_speeding"])
        + (w_power * d["r_power_total"])
        + (w_camera * d["r_camera_total"])
        + (w_fatigue * d["r_fatigue_total"])
        + (w_distraction * d["r_distraction_total"])
        + (w_situational * d["r_situational_total"])
    )

    d["insufficient_exposure_flag"] = (
        d["drive_hours"].fillna(0.0) < INSUFFICIENT_EXPOSURE_HOURS
    ).astype("int64")

    ranked_mask = (d["insufficient_exposure_flag"] == 0)

    if ranked_mask.any():
        q1 = (d.loc[ranked_mask, "ubi_proxy_score"].quantile(0.33))
        q2 = (d.loc[ranked_mask, "ubi_proxy_score"].quantile(0.67))
    else:
        q1 = 0.0
        q2 = 0.0

    def lab(row):
        if (row["insufficient_exposure_flag"] == 1):
            return "Insufficient Exposure"
        x = row["ubi_proxy_score"]
        if (x <= q1):
            return "Low"
        if (x <= q2):
            return "Medium"
        return "High"

    d["vehicle_behaviour_class"] = (d.apply(lab, axis=1))

    keep = [
        "vehicle_id",
        time_col,
        "window_end",
        "drive_hours",
        "distance_km",
        "insufficient_exposure_flag",
        "harsh_episode_count",
        "short_speeding_episode_count",
        "long_speeding_episode_count",
        "speeding_episode_count",
        "power_violation_episode_count",
        "camera_obstruction_episode_count",
        "fatigue_episode_count",
        "driver_distraction_episode_count",
        "situational_risk_episode_count",
        "crash_count",
        "ubi_proxy_score",
        "vehicle_behaviour_class",
    ]
    keep = [c for c in keep if (c in d.columns)]
    return (d[keep].sort_values(["vehicle_id", time_col]).reset_index(drop=True))

def class_definition_table(scored: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "drive_hours",
        "harsh_episode_count",
        "short_speeding_episode_count",
        "long_speeding_episode_count",
        "power_violation_episode_count",
        "camera_obstruction_episode_count",
        "fatigue_episode_count",
        "driver_distraction_episode_count",
        "situational_risk_episode_count",
        "ubi_proxy_score",
        "crash_count",
    ]
    cols = [c for c in cols if (c in scored.columns)]

    out = (
        scored.groupby("vehicle_behaviour_class", as_index=False)[cols]
        .mean(numeric_only=True)
        .sort_values("ubi_proxy_score")
        .reset_index(drop=True)
    )
    out.insert(0, "Vehicle Behaviour Class", out.pop("vehicle_behaviour_class"))
    return out

# =========================
# MASTER WRAPPER
# =========================
def run_driver_ranking_pipeline(raw_df: pd.DataFrame, show_output: bool = True) -> dict:
    # =========================
    # PREP
    # =========================
    df = (raw_df.copy())

    df["event_ts"] = (pd.to_datetime(df["event_ts"], utc=True, errors="coerce"))
    df["event_description"] = (df["event_description"].astype("string").str.strip())
    df["odometer"] = (pd.to_numeric(df.get("odometer", np.nan), errors="coerce"))
    df = (df.dropna(subset=["vehicle_id", "event_ts", "event_description"]).copy())
    df["vehicle_id"] = (df["vehicle_id"].astype("int64", errors="ignore"))
    df = (df.sort_values(["vehicle_id", "event_ts"]).reset_index(drop=True))
    df["day"] = (df["event_ts"].dt.floor("D"))
    desc_lower = (df["event_description"].astype("string").str.lower())

    # =========================
    # MAPPING APPENDIX
    # =========================
    taxonomy_df = (
        df[["event_description"]]
        .drop_duplicates()
        .assign(
            behaviour_category=lambda x: x["event_description"].map(lambda v: _map_event_to_proxy_bucket(v)[0]),
            feature_used=lambda x: x["event_description"].map(lambda v: _map_event_to_proxy_bucket(v)[1]),
        )
        .sort_values(["behaviour_category", "feature_used", "event_description"])
        .reset_index(drop=True)
    )

    if show_output:
        print("Unique event_description values mapped into the updated proxy taxonomy")
        display(taxonomy_df.head(200))

    # =========================
    # DRIVING TIME + DISTANCE
    # =========================
    v = (df[["vehicle_id", "event_ts", "odometer"]].copy())
    v["next_ts"] = (v.groupby("vehicle_id")["event_ts"].shift(-1))
    v["next_odo"] = (v.groupby("vehicle_id")["odometer"].shift(-1))
    v["delta_s"] = ((v["next_ts"] - v["event_ts"]).dt.total_seconds())

    intervals = (
        v.loc[
            (v["delta_s"].notna())
            & (v["delta_s"] > 0)
            & (v["delta_s"] <= SESSION_GAP_SECONDS)
        ].copy()
    )

    intervals = (
        intervals.rename(
            columns={
                "event_ts": "t0",
                "next_ts": "t1",
                "odometer": "odo0",
                "next_odo": "odo1",
            }
        )
    )

    intervals["delta_km"] = (
        np.where(
            (intervals["odo0"].notna()) & (intervals["odo1"].notna()),
            (intervals["odo1"] - intervals["odo0"]),
            np.nan,
        )
    )

    intervals.loc[intervals["delta_km"] < 0, "delta_km"] = (np.nan)

    rows = []
    for r in (intervals.itertuples(index=False)):
        parts = (_split_interval_by_day(r.t0, r.t1))
        if (not parts):
            continue

        total_s = ((r.t1 - r.t0).total_seconds())
        if (total_s <= 0):
            continue

        for day_start, p0, p1 in parts:
            dur_s = ((p1 - p0).total_seconds())
            night_s = (_night_overlap_seconds(p0, p1))

            if (pd.notna(r.delta_km)):
                km_part = (float(r.delta_km) * (dur_s / total_s))
                night_km_part = ((km_part * (night_s / dur_s)) if (dur_s > 0) else 0.0)
            else:
                km_part = (np.nan)
                night_km_part = (np.nan)

            rows.append((r.vehicle_id, day_start, dur_s, night_s, km_part, night_km_part))

    drive_daily = (
        pd.DataFrame(
            rows,
            columns=[
                "vehicle_id",
                "day",
                "drive_seconds",
                "night_drive_seconds",
                "distance_km_part",
                "night_distance_km_part",
            ],
        )
    )

    if (drive_daily.empty):
        drive_daily = pd.DataFrame(
            columns=[
                "vehicle_id",
                "day",
                "drive_seconds",
                "night_drive_seconds",
                "distance_km_part",
                "night_distance_km_part",
            ]
        )

    drive_daily = (
        drive_daily.groupby(["vehicle_id", "day"], as_index=False)
        .agg(
            drive_seconds=("drive_seconds", "sum"),
            night_drive_seconds=("night_drive_seconds", "sum"),
            distance_km=("distance_km_part", "sum"),
            night_distance_km=("night_distance_km_part", "sum"),
        )
    )

    known_km_flags = (
        drive_daily[["vehicle_id", "day"]]
        .merge(
            intervals.assign(day=intervals["t0"].dt.floor("D"))[["vehicle_id", "day", "delta_km"]],
            on=["vehicle_id", "day"],
            how="left",
        )
    )

    known_km_any = (
        known_km_flags.groupby(["vehicle_id", "day"], as_index=False)["delta_km"]
        .apply(lambda s: s.notna().any())
        .rename(columns={"delta_km": "has_km"})
    )

    drive_daily = (drive_daily.merge(known_km_any, on=["vehicle_id", "day"], how="left"))
    drive_daily.loc[(drive_daily["has_km"] == False), ["distance_km", "night_distance_km"]] = (np.nan)
    drive_daily = (drive_daily.drop(columns=["has_km"]))

    drive_daily["drive_minutes"] = (drive_daily["drive_seconds"] / 60.0)
    drive_daily["drive_hours"] = (drive_daily["drive_seconds"] / 3600.0)
    drive_daily["night_ratio"] = (
        np.where(
            (drive_daily["drive_seconds"] > 0),
            (drive_daily["night_drive_seconds"] / drive_daily["drive_seconds"]),
            0.0,
        )
    )

    if show_output:
        print("Per vehicle_id per day estimated driving exposure kept for eligibility and internal analysis")
        display(drive_daily.sort_values(["vehicle_id", "day"]).head(50))

    # =========================
    # RAW EVENT COUNTS (appendix only)
    # =========================
    counts_daily = (df[["vehicle_id", "day"]].drop_duplicates().copy())

    for substr_map in [
        HARSH_SUBSTRINGS,
        SPEEDING_SUBSTRINGS,
        POWER_SUBSTRINGS,
        CAMERA_SUBSTRINGS,
        FATIGUE_SUBSTRINGS,
        DISTRACTION_SUBSTRINGS,
        SITUATIONAL_SUBSTRINGS,
        CRASH_SUBSTRINGS,
    ]:
        counts_daily = (_add_raw_counts(df, desc_lower, counts_daily, substr_map))

    raw_count_cols = [col for _, col in ALL_EVENT_COUNT_MAPS]

    for c in raw_count_cols:
        if (c not in counts_daily.columns):
            counts_daily[c] = 0
        counts_daily[c] = (counts_daily[c].fillna(0).astype("int64"))

    # =========================
    # EPISODE COUNTS (manager feedback)
    # =========================
    harsh_episode_daily = (_episode_counts_from_mask(
        df,
        _build_match_mask(desc_lower, HARSH_SUBSTRINGS),
        HARSH_EPISODE_WINDOW_SECONDS,
        "harsh_episode_count",
    ))

    speeding_episode_daily = (_speeding_episode_breakdown(df, desc_lower))

    power_episode_daily = (_episode_counts_from_mask(
        df,
        _build_match_mask(desc_lower, POWER_SUBSTRINGS),
        POWER_EPISODE_WINDOW_SECONDS,
        "power_violation_episode_count",
    ))

    camera_episode_daily = (_episode_counts_from_mask(
        df,
        _build_match_mask(desc_lower, CAMERA_SUBSTRINGS),
        CAMERA_EPISODE_WINDOW_SECONDS,
        "camera_obstruction_episode_count",
    ))

    fatigue_episode_daily = (_episode_counts_from_mask(
        df,
        _build_match_mask(desc_lower, FATIGUE_SUBSTRINGS),
        FATIGUE_EPISODE_WINDOW_SECONDS,
        "fatigue_episode_count",
    ))

    distraction_episode_daily = (_episode_counts_from_mask(
        df,
        _build_match_mask(desc_lower, DISTRACTION_SUBSTRINGS),
        DISTRACTION_EPISODE_WINDOW_SECONDS,
        "driver_distraction_episode_count",
    ))

    situational_episode_daily = (_episode_counts_from_mask(
        df,
        _build_match_mask(desc_lower, SITUATIONAL_SUBSTRINGS),
        SITUATIONAL_EPISODE_WINDOW_SECONDS,
        "situational_risk_episode_count",
    ))

    # =========================
    # DAILY FEATURE VECTORS
    # =========================
    features_daily = (counts_daily.merge(drive_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(harsh_episode_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(speeding_episode_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(power_episode_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(camera_episode_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(fatigue_episode_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(distraction_episode_daily, on=["vehicle_id", "day"], how="left"))
    features_daily = (features_daily.merge(situational_episode_daily, on=["vehicle_id", "day"], how="left"))

    fill0 = {
        "drive_seconds": 0.0,
        "night_drive_seconds": 0.0,
        "drive_minutes": 0.0,
        "drive_hours": 0.0,
        "night_ratio": 0.0,
        "harsh_episode_count": 0,
        "short_speeding_episode_count": 0,
        "long_speeding_episode_count": 0,
        "speeding_episode_count": 0,
        "power_violation_episode_count": 0,
        "camera_obstruction_episode_count": 0,
        "fatigue_episode_count": 0,
        "driver_distraction_episode_count": 0,
        "situational_risk_episode_count": 0,
    }
    features_daily = (features_daily.fillna(fill0))
    features_daily["distance_km_filled"] = (features_daily["distance_km"].fillna(0.0))

    # Raw grouped totals kept for analysis
    features_daily["harsh_total_count"] = (
        features_daily["harsh_corner_count"]
        + features_daily["harsh_braking_count"]
        + features_daily["harsh_accel_count"]
    )

    features_daily["fatigue_total_count"] = (
        features_daily["eye_closed_count"]
        + features_daily["yawn_count"]
        + features_daily["fatigue_count"]
    )

    features_daily["driver_distraction_total_count"] = (
        features_daily["distraction_count"]
        + features_daily["phone_use_count"]
        + features_daily["smoke_count"]
        + features_daily["Driver_Seatbelt_not_on"]
    )

    features_daily["situational_risk_total_count"] = (
        features_daily["headway_monitor_count"]
        + features_daily["lane_departure_count"]
        + features_daily["forward_collision_count"]
        + features_daily["ped_collision_count"]
    )

    features_daily = (features_daily.sort_values(["vehicle_id", "day"]).reset_index(drop=True))

    if show_output:
        print("Daily features with episode aggregation applied per manager feedback")
        display(features_daily.head(50))

    # =========================
    # WEEKLY AND MONTHLY AGG
    # =========================
    features_weekly = (_aggregate(features_daily, "week"))
    features_monthly = (_aggregate(features_daily, "month"))

    if show_output:
        print("Weekly aggregation with episode-based behaviour metrics")
        display(features_weekly.head(50))
        print("Monthly aggregation with episode-based behaviour metrics")
        display(features_monthly.head(50))

    # =========================
    # SCORING + CLASSES
    # =========================
    features_daily = (features_daily.copy())
    features_daily["window_start"] = (features_daily["day"])
    features_daily["window_end"] = (features_daily["day"] + pd.Timedelta(days=1))

    scored_daily = (add_score_and_class(features_daily, "window_start"))
    scored_weekly = (add_score_and_class(features_weekly, "window_start"))
    scored_monthly = (add_score_and_class(features_monthly, "window_start"))

    if show_output:
        print("Daily scored output with Insufficient Exposure and episode-based scoring")
        display(scored_daily.head(50))
        print("Weekly scored output with Insufficient Exposure and episode-based scoring")
        display(scored_weekly.head(50))
        print("Monthly scored output with Insufficient Exposure and episode-based scoring")
        display(scored_monthly.head(50))

    # =========================
    # MONTHLY CRASH SUMMARY
    # =========================
    crash_monthly = (
        scored_monthly.groupby(["window_start"], as_index=False)
        .agg(
            crash_events=("crash_count", "sum"),
            vehicles_with_crash=("crash_count", lambda s: int((s > 0).sum())),
            total_vehicles=("vehicle_id", "nunique"),
        )
    )
    crash_monthly["pct_vehicles_with_crash"] = (
        np.where(
            crash_monthly["total_vehicles"] > 0,
            (crash_monthly["vehicles_with_crash"] / crash_monthly["total_vehicles"]) * 100.0,
            0.0,
        )
    )

    if show_output:
        print("Monthly crash summary (analysis only)")
        display(crash_monthly)

    crash_monthly_by_vehicle = (
        scored_monthly.loc[scored_monthly["crash_count"] > 0, ["vehicle_id", "window_start", "crash_count"]]
        .sort_values(["window_start", "crash_count"], ascending=[True, False])
        .reset_index(drop=True)
    )

    if show_output:
        print("Monthly crashes by vehicle (analysis only)")
        display(crash_monthly_by_vehicle.head(200))

    # =========================
    # CLASS DEFINITION TABLE
    # =========================
    class_defs_daily = (class_definition_table(scored_daily))
    class_defs_weekly = (class_definition_table(scored_weekly))
    class_defs_monthly = (class_definition_table(scored_monthly))

    if show_output:
        print("Mean feature values per behavioural class (daily)")
        display(class_defs_daily)
        print("Mean feature values per behavioural class (weekly)")
        display(class_defs_weekly)
        print("Mean feature values per behavioural class (monthly)")
        display(class_defs_monthly)

    # =========================
    # SANITY CHECKS
    # =========================
    if show_output:
        print("Session gap seconds:", SESSION_GAP_SECONDS)
        print("Insufficient exposure cutoff (hours):", INSUFFICIENT_EXPOSURE_HOURS)
        print("Unique vehicles:", df["vehicle_id"].nunique())
        print("Unique event_descriptions:", df["event_description"].nunique())

    top_events = (
        df["event_description"]
        .value_counts()
        .head(30)
        .rename_axis("event_description")
        .reset_index(name="count")
    )

    if show_output:
        display(top_events)

    return {
        "df": df,
        "taxonomy_df": taxonomy_df,
        "drive_daily": drive_daily,
        "counts_daily": counts_daily,
        "features_daily": features_daily,
        "features_weekly": features_weekly,
        "features_monthly": features_monthly,
        "scored_daily": scored_daily,
        "scored_weekly": scored_weekly,
        "scored_monthly": scored_monthly,
        "crash_monthly": crash_monthly,
        "crash_monthly_by_vehicle": crash_monthly_by_vehicle,
        "class_defs_daily": class_defs_daily,
        "class_defs_weekly": class_defs_weekly,
        "class_defs_monthly": class_defs_monthly,
        "top_events": top_events,
    }

# =========================
# RUN UPDATED PIPELINE
# =========================
base_outputs = run_driver_ranking_pipeline(raw_df, show_output=True)

df = base_outputs["df"]
taxonomy_df = base_outputs["taxonomy_df"]
drive_daily = base_outputs["drive_daily"]
counts_daily = base_outputs["counts_daily"]
features_daily = base_outputs["features_daily"]
features_weekly = base_outputs["features_weekly"]
features_monthly = base_outputs["features_monthly"]
scored_daily = base_outputs["scored_daily"]
scored_weekly = base_outputs["scored_weekly"]
scored_monthly = base_outputs["scored_monthly"]
crash_monthly = base_outputs["crash_monthly"]
crash_monthly_by_vehicle = base_outputs["crash_monthly_by_vehicle"]
class_defs_daily = base_outputs["class_defs_daily"]
class_defs_weekly = base_outputs["class_defs_weekly"]
class_defs_monthly = base_outputs["class_defs_monthly"]
top_events = base_outputs["top_events"]

How to interpret it

night_ratio = 0.0
The vehicle was never driven at night in that window.

night_ratio = 0.25
About 25% of its driving time was at night.

night_ratio = 1.0
All driving happened at night.

In [ ]:
# Looking at a long speeding episode

In [ ]:

# Identify speeding events
raw_df["event_ts"] = pd.to_datetime(raw_df["event_ts"], errors="coerce")
speed_mask = raw_df["event_description"].str.contains("Speeding", case=False, na=False)

speed_df = (
    raw_df.loc[speed_mask, ["vehicle_id", "event_ts", "event_description"]]
    .sort_values(["vehicle_id", "event_ts"])
    .copy()
)

# Build speeding episodes (same logic as pipeline)
speed_df["prev_ts"] = speed_df.groupby("vehicle_id")["event_ts"].shift(1)
speed_df["gap_s"] = (speed_df["event_ts"] - speed_df["prev_ts"]).dt.total_seconds()

WINDOW = 5 * 60  # 5 minutes

speed_df["new_episode"] = (
    speed_df["prev_ts"].isna() | (speed_df["gap_s"] > WINDOW)
).astype(int)

speed_df["episode_id"] = speed_df.groupby("vehicle_id")["new_episode"].cumsum()

# Count events in each episode
episode_sizes = (
    speed_df.groupby(["vehicle_id", "episode_id"])
    .size()
    .reset_index(name="events_in_episode")
)

# Long speeding = more than 1 event in episode
long_eps = episode_sizes.loc[episode_sizes["events_in_episode"] > 1]

vehicle_id = 523824244
speed_df = speed_df.loc[speed_df["vehicle_id"] == vehicle_id]


# Pull the raw events for those long speeding episodes
long_speeding_events = speed_df.merge(
    long_eps[["vehicle_id", "episode_id"]],
    on=["vehicle_id", "episode_id"],
    how="inner"
).sort_values(["vehicle_id", "event_ts"])



long_speeding_events

In [ ]:
class_defs_weekly

In [ ]:
worst = (
    scored_weekly
    .loc[scored_weekly["insufficient_exposure_flag"] == 0]
    .sort_values(by="ubi_proxy_score", ascending=False)
)

worst.head(10)

In [ ]:
best = (
    scored_weekly
    .loc[scored_weekly["insufficient_exposure_flag"] == 0]
    .sort_values(by="ubi_proxy_score", ascending=True)
)

best.head(10)

In [ ]:
scored_monthly.sort_values(by="crash_count", ascending=True)

# Cameras

In [ ]:
# I cannot provide this. Rebuild to provide VLM clips

video_df

# Get context from URLs

# VLM STUFFFF

In [ ]:
"""
CELL 1 — CONFIG & CONSTANTS
Run this cell first. All tunable parameters live here.
"""

# =========================
# OLLAMA CONFIG
# =========================
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost") #Add your corrected localhost. 
OLLAMA_MODEL    = os.getenv("OLLAMA_MODEL",    "qwen2.5vl:7b")
OLLAMA_TIMEOUT  = int(os.getenv("OLLAMA_TIMEOUT", "600"))

# =========================
# PERFORMANCE TUNING
# =========================
FRAME_SIZE_MAX = int(os.getenv("FRAME_SIZE_MAX", "576"))
JPEG_QUALITY   = int(os.getenv("JPEG_QUALITY",   "78"))
MAX_TOKENS     = int(os.getenv("MAX_TOKENS",      "512"))
STOP_MARKER    = "\nEND_JSON"

# =========================
# FRAMES
# =========================
NUM_FRAMES = int(os.getenv("NUM_FRAMES", "12"))
NUM_FRAMES = max(2, min(12, NUM_FRAMES))

MOTION_FRAMES_PER_SEC = float(os.getenv("MOTION_FRAMES_PER_SEC", "3"))
MOTION_MIN_FRAMES     = int(os.getenv("MOTION_MIN_FRAMES",       "20"))
MOTION_MAX_FRAMES     = int(os.getenv("MOTION_MAX_FRAMES",       "48"))

# =========================
# OUTPUT CONTROLS
# =========================
OMIT_EMPTY_LISTS = os.getenv("OMIT_EMPTY_LISTS", "1") == "1"

# =========================
# DEBUG / FILE PATHS
# =========================
OUTPUT_BASE_DIR      = "debug_frames_qwen"
DEBUG_FRAMES_DIR     = OUTPUT_BASE_DIR          # alias kept for compatibility
SAVE_DEBUG_FRAMES    = True
SOURCE_META_FILENAME = "source_meta.json"
RESULT_FILENAME      = "result.json"
ERROR_FILENAME       = "error.txt"

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

# =========================
# EVENT CODES
# =========================
EVENT_CODE = {
    "OBSTRUCTION_VIEW":       4,
    "SEATBELT_OFF_DRIVER":    5,
    "CAMERA_COVERED":         6,
    "PHONE_DRIVER":           10,
    "DISTRACTION":            11,
    "FATIGUE_DRIVER":         13,
    "SMOKING_DRIVER":         15,
    "DRIVER_FACE_OBSTRUCTED": 66,
    "CAMERA_COVERED_REVIEW":  67,   # tier 2 — possible covering, VLM ran but flagged for review
}
CODE_EVENT = {v: k for k, v in EVENT_CODE.items()}

# =========================
# LABELLED EVENTS
# =========================
LABELLED_EVENTS = {
    "v_cam_covered",
    "v_distraction",
    "v_eye_closed",
    "v_fatigue",
    "v_phone",
    "v_yawn",
    "v_smoke",
    "SEATBELT_D_OFF",
}

# =========================
# QUALITY GATES
# =========================
DARK_MEAN_THRESH      = float(os.getenv("DARK_MEAN_THRESH",      "18.0"))
BAD_FRAME_RATIO_THRESH = float(os.getenv("BAD_FRAME_RATIO_THRESH", "0.75"))

# Tier 1 — definitely covered (solid object blocking lens)
SMEAR_BLOB_RATIO_T     = float(os.getenv("SMEAR_BLOB_RATIO_T",   "0.90"))  # was 0.55
SMEAR_GRAD_P75_T       = float(os.getenv("SMEAR_GRAD_P75_T",     "5.0"))   # was 14.0
SMEAR_DARK_RATIO_MAX   = float(os.getenv("SMEAR_DARK_RATIO_MAX", "0.85"))
SMEAR_BRIGHT_RATIO_MAX = float(os.getenv("SMEAR_BRIGHT_RATIO_MAX", "0.85"))

# Tier 2 — possibly covered / blurry (cream, dirty lens, beads)
# lap_roi below this = low texture = possibly covered, send to VLM for review
LAP_ROI_COVERED_T = float(os.getenv("LAP_ROI_COVERED_T", "100.0"))

ROI_CROP      = (0.18, 0.20, 0.10, 0.10)
BLUR_ROI_CROP = (0.25, 0.25, 0.35, 0.10)

COVERED_ENTROPY_T    = float(os.getenv("COVERED_ENTROPY_T",    "3.2"))
COVERED_EDGE_DENSITY_T = float(os.getenv("COVERED_EDGE_DENSITY_T", "0.025"))
COVERED_RATIO_T      = float(os.getenv("COVERED_RATIO_T",      "0.70"))
BLUR_ROI_LAP_VAR_T   = float(os.getenv("BLUR_ROI_LAP_VAR_T",  "20.0"))
COVERED_ROI_STD_T    = float(os.getenv("COVERED_ROI_STD_T",   "12.0"))

# =========================
# RECHECK: SEATBELT + PHONE
# =========================
RECHECK_ENABLED                  = os.getenv("RECHECK_ENABLED", "1") == "1"
REVIEW_ON_SEATBELT_U_WHEN_MOVING = os.getenv("REVIEW_ON_SEATBELT_U_WHEN_MOVING", "1") == "1"
RECHECK_ROI_CROPS    = [(0.28, 0.18, 0.02, 0.55), (0.22, 0.22, 0.00, 0.60)]
RECHECK_FRAME_MAX    = int(os.getenv("RECHECK_FRAME_MAX",    "640"))
RECHECK_JPEG_QUALITY = int(os.getenv("RECHECK_JPEG_QUALITY", "90"))
RECHECK_MAX_TOKENS   = int(os.getenv("RECHECK_MAX_TOKENS",   "220"))

# =========================
# FACE OBSTRUCTION 66
# =========================
FACE66_ENABLED           = os.getenv("FACE66_ENABLED", "1") == "1"
FACE66_MIN_USABLE_FRAMES = int(os.getenv("FACE66_MIN_USABLE_FRAMES", "3"))
FACE66_NOFACE_RATIO_T    = float(os.getenv("FACE66_NOFACE_RATIO_T",  "0.70"))
FACE66_LOWINFO_RATIO_T   = float(os.getenv("FACE66_LOWINFO_RATIO_T", "0.35"))
FACE66_ZONE_STD_T        = float(os.getenv("FACE66_ZONE_STD_T",      "18.0"))
FACE66_ZONE_ENT_T        = float(os.getenv("FACE66_ZONE_ENT_T",      "4.2"))

# =========================
# MOTION ROI
# =========================
MOTION_WINDOW_ROI = (
    float(os.getenv("MOTION_WINDOW_TOP",    "0.10")),
    float(os.getenv("MOTION_WINDOW_BOTTOM", "0.18")),
    float(os.getenv("MOTION_WINDOW_LEFT",   "0.00")),
    float(os.getenv("MOTION_WINDOW_RIGHT",  "0.52")),
)
MOTION_WINDSCREEN_ROI = (
    float(os.getenv("MOTION_WINDSCREEN_TOP",    "0.05")),
    float(os.getenv("MOTION_WINDSCREEN_BOTTOM", "0.52")),
    float(os.getenv("MOTION_WINDSCREEN_LEFT",   "0.08")),
    float(os.getenv("MOTION_WINDSCREEN_RIGHT",  "0.35")),
)
MOTION_USE_WINDSCREEN = os.getenv("MOTION_USE_WINDSCREEN", "1") == "1"

MOTION_PAIR_LOW_T  = float(os.getenv("MOTION_PAIR_LOW_T",  "0.010"))
MOTION_PAIR_HIGH_T = float(os.getenv("MOTION_PAIR_HIGH_T", "0.028"))

MOTION_STOPPED_MAX_P75          = float(os.getenv("MOTION_STOPPED_MAX_P75",          "0.0135"))
MOTION_STOPPED_MAX_ACTIVE_RATIO = float(os.getenv("MOTION_STOPPED_MAX_ACTIVE_RATIO", "0.12"))
MOTION_STOPPED_MAX_FIRST_LAST   = float(os.getenv("MOTION_STOPPED_MAX_FIRST_LAST",   "0.018"))

MOTION_MOVING_MIN_ACTIVE_RATIO = float(os.getenv("MOTION_MOVING_MIN_ACTIVE_RATIO", "0.30"))
MOTION_MOVING_MIN_P75          = float(os.getenv("MOTION_MOVING_MIN_P75",          "0.01"))
MOTION_MOVING_MIN_FIRST_LAST   = float(os.getenv("MOTION_MOVING_MIN_FIRST_LAST",   "0.030"))
MOTION_MIN_USABLE_PAIRS        = int(os.getenv("MOTION_MIN_USABLE_PAIRS",          "5"))

print("✓ Cell 1: Config loaded")
print(f"  Model: {OLLAMA_MODEL}  |  Ollama: {OLLAMA_BASE_URL}")
print(f"  NUM_FRAMES={NUM_FRAMES}  |  FRAME_SIZE_MAX={FRAME_SIZE_MAX}")
print(f"  Output dir: {OUTPUT_BASE_DIR}")

In [ ]:
"""
CELL 2 — PROMPTS
Depends on: Cell 1 (NUM_FRAMES)
"""

PROMPT_TEMPLATE = """
You are analyzing {num_frames} frames from the same clip in chronological order.
These frames show ONLY the driver side (left half of cabin camera).

Return STRICT JSON then END_JSON.

Rules:
- Do not guess. If not clearly visible, use the rules below. For eatdrink you must output y or n.
- Do not infer from context. Use only what is visible.
- No passenger analysis.

Output JSON:
{{
  "driver_vis": 0|1|2,
  "seatbelt": "y"|"n"|"u",
  "phone": "y"|"n"|"u",
  "distraction": "y"|"n"|"u",
  "fatigue": "y"|"n"|"u",
  "smoking": "y"|"n"|"u",
  "eatdrink": "y"|"n",
  "drinking": "y"|"n"|"u",
  "alcohol": "y"|"n"|"u",
  "seatbelt_f": [..],
  "phone_f": [..],
  "distraction_f": [..],
  "fatigue_f": [..],
  "smoking_f": [..],
  "eatdrink_f": [..],
  "why": "<very short>"
}}

Meanings:
- driver_vis: 0=unknown, 1=driver clearly visible, 2=driver not visible

Seatbelt:
- seatbelt: y ONLY if a continuous diagonal seatbelt strap clearly crosses the driver's chest from shoulder toward hip (properly worn).
- seatbelt: n if the driver's chest/torso is visible and no strap crosses the chest.
- seatbelt: u only if the torso/chest area cannot be seen or is heavily occluded.

Important:
- Only mark y when a clear strap path across the chest is visible.
- Do NOT treat clothing folds, shirt seams, shadows, sunlight bands, seat stitching, or door frame edges as seatbelts.
- If the torso is visible and no strap crosses the chest, output n (not u).

Phone:
- phone: y ONLY if a clearly visible phone/device is present in the driver's hand or at the ear, with a clear rectangular phone-like object or obvious screen/device shape.
- phone: n ONLY if the relevant hands/face area are clearly visible and no phone/device is present.
- phone: u if the device itself cannot be clearly seen.
- Do NOT mark phone=y for scratching head, touching face, rubbing temple, adjusting hair, resting hand on cheek, covering mouth, or any generic hand-near-face gesture unless the phone/device itself is clearly visible.
- Do NOT confuse reflections, fingers, sleeves, cigarette/vape, pens, straws, or other small objects with a phone.

Distraction:
- distraction: y ONLY if clearly distracted (looking down for sustained moment, interacting with items).
- otherwise u if unclear, or n if clearly attentive.

Fatigue:
- fatigue: y ONLY if you clearly see an obvious yawn (mouth wide open for 2 or more frames in a row) OR eyes fully closed for a moment OR clear head nod with eyes closing.
- otherwise u if face unclear, sunglasses, lighting poor, blur.
- n only if face clearly visible and none of the above is present.

Smoking:
- smoking: y ONLY if you clearly see cigarette/vape at mouth OR visible smoke plume OR clear exhale with device visible.
- Do NOT confuse lollipops, straws, pens, white sticks, or reflections with cigarettes.
- otherwise u if unclear.
- n only if hands/face clearly visible and no smoking device appears.

Eating/Drinking (binary):
- eatdrink must be y or n (no u). Use y ONLY if you clearly see eating OR drinking (food at mouth, chewing with food visible, bottle/cup at mouth).
- otherwise eatdrink=n.

Drinking detail:
- drinking: y only if clearly drinking (container at mouth).
- alcohol: y only if the beverage is clearly alcoholic (visible beer bottle/can, wine/spirits bottle, branded alcohol, obvious label). Otherwise n or u if unclear.

Frame lists:
- *_f are frame indexes (1..{num_frames}) supporting your decision. Empty if unknown.

Hard constraints:
- If driver is not visible (driver_vis==2), set seatbelt="u", phone="u", distraction="u", fatigue="u", smoking="u", drinking="u", alcohol="u", and set eatdrink="n".
- If image is unclear, choose "u" for uncertain fields.
""".strip()

RECHECK_SEATBELT_PHONE_PROMPT = """
You are checking ONLY the driver seatbelt and phone use from multiple cropped torso images from the same clip.
These crops come from the driver side. Ignore passengers.

Return STRICT JSON then END_JSON.

Rules:
- Seatbelt "y" ONLY if a continuous diagonal strap clearly crosses the driver's chest from shoulder toward hip.
- Seatbelt "n" if the torso/chest area is visible and no strap crosses the chest.
- Seatbelt "u" only if the torso/chest area cannot be seen.

Important:
- Do NOT confuse clothing folds, shirt seams, shadows, reflections, seat stitching, or door frame edges with a seatbelt.
- Phone "y" ONLY if you clearly see a phone/device in the driver's hand or being used, with a clear device shape. Not just the drivers hand in a phone like position.
- Phone "n" ONLY if you clearly see the relevant hands/area and no phone/device is present.
- Phone "u" if the device itself cannot be clearly seen.
- Do NOT mark phone=y for scratching head, touching face, rubbing temple, adjusting hair, resting hand on cheek, or other hand-near-face actions unless the phone/device itself is clearly visible.
- Do NOT confuse reflections, fingers, sleeves, cigarettes, pens, or white sticks with a phone.

Output JSON:
{
  "seatbelt": "y"|"n"|"u",
  "phone": "y"|"n"|"u",
  "seatbelt_f": [..],
  "phone_f": [..],
  "why": "<very short>"
}
""".strip()

SMOKING_ONLY_PROMPT = f"""
You are analyzing {NUM_FRAMES} frames from the same clip in chronological order.
These frames show ONLY the driver side (left half of cabin camera). Ignore passengers.

Return STRICT JSON then END_JSON.

Output JSON:
{{
  "driver_vis": 0|1|2,
  "smoking": "y"|"n"|"u",
  "smoking_f": [..],
  "why": "<very short>"
}}

Rules:
- Ignore all on-screen text, labels, timestamps, watermarks, event names, and overlays. Never use words shown on the image as evidence.
- smoking=y ONLY if you clearly see a real cigarette or vape device, and it is clearly being used by the driver.
- A real cigarette means a clearly visible cigarette-like stick. A vape means a clearly visible vape device, e-cigarette, or similar smoking device.
- smoking=y requires at least one of these:
  1. the cigarette or vape is clearly at the mouth, OR
  2. a clear smoke plume or exhale is visible, OR
  3. the same clearly identified smoking device is visible across multiple frames and is clearly being used by the driver.
- Do NOT mark smoking=y from pose alone, mouth shape alone, pursed lips alone, sucking expression alone, hand position alone, or a thin object alone.
- Do NOT confuse lollipops, straws, pens, fingers, nails, food sticks, reflections, jewellery, clothing edges, braids, wires, or other small objects with cigarettes or vapes.
- If there is a thin object at the mouth but it is not clearly identifiable as a cigarette or vape, set smoking=n.
- If the object might be a lollipop, straw, pen, or any non-smoking item, set smoking=n.
- smoking=n if the face or mouth area is visible and there is no clearly identifiable cigarette, vape, or smoke.
- smoking=u only if the driver or mouth area is too unclear to assess.
- If driver is not visible, set driver_vis=2 and smoking="u".
- smoking_f must contain only frame indexes that directly support smoking=y. If smoking is not y, return an empty list.
- Be highly conservative. Only use smoking=y when you are very confident.

Decision standard:
- When uncertain, choose n, not y.
""".strip()

PHONE_ONLY_PROMPT = f"""
You are analyzing {NUM_FRAMES} frames from the same clip in chronological order.
These frames show ONLY the driver side (left half of cabin camera). Ignore passengers.

Return STRICT JSON then END_JSON.

Output JSON:
{{
  "driver_vis": 0|1|2,
  "phone": "y"|"n"|"u",
  "phone_f": [..],
  "why": "<very short>"
}}

Rules:
- phone=y ONLY if a clearly visible phone/device is present in the driver's hand or at the ear, with a clear rectangular phone-like object or obvious screen/device shape.
- phone=n only if the relevant hands/face area are clearly visible and no phone/device is present.
- phone=u if the device itself cannot be clearly seen.
- Do NOT mark phone=y for scratching head, touching face, rubbing temple, adjusting hair, resting hand on cheek, or any generic hand-near-face gesture unless the phone/device itself is clearly visible.
- Do NOT confuse reflections, fingers, sleeves, cigarette/vape, pens, or other small objects with a phone.
- If driver is not visible, set driver_vis=2 and phone="u".
""".strip()

SEATBELT_ONLY_PROMPT = f"""
You are analyzing {NUM_FRAMES} frames from the same clip in chronological order.
These frames show ONLY the driver side (left half of cabin camera). Ignore passengers.

Return STRICT JSON then END_JSON.

Output JSON:
{{
  "driver_vis": 0|1|2,
  "seatbelt": "y"|"n"|"u",
  "seatbelt_f": [..],
  "why": "<very short>"
}}

Rules:
- seatbelt=y ONLY if a continuous diagonal seatbelt strap clearly crosses the driver's chest from shoulder toward hip.
- seatbelt=n if the driver's chest/torso is visible and no strap crosses the chest.
- seatbelt=u only if the torso/chest area cannot be seen or is heavily occluded.
- Do NOT confuse clothing folds, seams, shadows, sunlight bands, stitching, or frame edges with a seatbelt.
- If driver is not visible, set driver_vis=2 and seatbelt="u".
""".strip()

DISTRACTION_ONLY_PROMPT = f"""
You are analyzing {NUM_FRAMES} frames from the same clip in chronological order.
These frames show ONLY the driver side (left half of cabin camera). Ignore passengers.

Return STRICT JSON then END_JSON.

Output JSON:
{{
  "driver_vis": 0|1|2,
  "distraction": "y"|"n"|"u",
  "distraction_f": [..],
  "why": "<very short>"
}}

Rules:
- distraction=y ONLY if clearly distracted, for example looking down for a sustained moment or interacting with items instead of driving attentively.
- distraction=n only if the driver is clearly visible and appears attentive.
- distraction=u if unclear.
- If driver is not visible, set driver_vis=2 and distraction="u".
""".strip()

FATIGUE_ONLY_PROMPT = f"""
You are analyzing {NUM_FRAMES} frames from the same clip in chronological order.
These frames show ONLY the driver side (left half of cabin camera). Ignore passengers.

Return STRICT JSON then END_JSON.

Output JSON:
{{
  "driver_vis": 0|1|2,
  "fatigue": "y"|"n"|"u",
  "fatigue_f": [..],
  "why": "<very short>"
}}

Rules:
- fatigue=y ONLY if you clearly see an obvious yawn (mouth wide open) OR eyes fully closed for a moment OR a clear head nod with eyes closing.
- fatigue=n only if the face is clearly visible and none of the above is present.
- fatigue=u if unclear, sunglasses, blur, darkness, or face not visible.
- If driver is not visible, set driver_vis=2 and fatigue="u".
""".strip()


def build_prompt() -> str:
    return PROMPT_TEMPLATE.format(num_frames=NUM_FRAMES)


print("✓ Cell 2: Prompts loaded")

In [ ]:
"""
CELL 3 — UTILITY HELPERS
Image utils, video sampling, quality metrics, motion detection.
Depends on: Cell 1 (all config constants)
"""

import base64
import io
import tempfile
import time
from typing import Any, Dict, List, Optional, Tuple

import av
import numpy as np
import requests
from PIL import Image
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout
from urllib3.exceptions import ProtocolError


# =========================
# DOWNLOAD WITH RETRY + RESUME
# =========================

def download_video(url: str, max_retries: int = 5) -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4")
    tmp_path = tmp.name
    tmp.close()

    for attempt in range(1, max_retries + 1):
        try:
            existing_size = os.path.getsize(tmp_path) if os.path.exists(tmp_path) else 0
            headers = {"Range": f"bytes={existing_size}-"} if existing_size > 0 else {}

            with requests.get(url, stream=True, timeout=60, headers=headers) as r:
                r.raise_for_status()
                mode = "ab" if existing_size > 0 else "wb"
                with open(tmp_path, mode) as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            return tmp_path

        except (ChunkedEncodingError, ProtocolError, ConnectionError, Timeout) as e:
            print(f"  Download attempt {attempt} failed: {e}")
            if attempt == max_retries:
                raise
            time.sleep(2 * attempt)

    return tmp_path


# =========================
# IMAGE UTILS
# =========================

def resize_image_keep_aspect(img: Image.Image, max_side: int) -> Image.Image:
    w, h = img.size
    m = max(w, h)
    if m <= max_side:
        return img
    scale = max_side / float(m)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    return img.resize((nw, nh), Image.BICUBIC)


def pil_to_data_url(img: Image.Image, quality: int = 80) -> str:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


def crop_frac(img: Image.Image, crop: Tuple[float, float, float, float]) -> Image.Image:
    w, h = img.size
    top_f, bot_f, left_f, right_f = crop
    x0 = int(w * left_f)
    x1 = int(w * (1.0 - right_f))
    y0 = int(h * top_f)
    y1 = int(h * (1.0 - bot_f))
    x0 = max(0, min(w - 2, x0));  x1 = max(x0 + 2, min(w, x1))
    y0 = max(0, min(h - 2, y0));  y1 = max(y0 + 2, min(h, y1))
    return img.crop((x0, y0, x1, y1))


def split_left_right(img: Image.Image) -> Tuple[Image.Image, Image.Image]:
    w, h = img.size
    mid = w // 2
    return img.crop((0, 0, mid, h)), img.crop((mid, 0, w, h))


def crop_frac_from_full_then_clip_to_left(
    full_img: Image.Image,
    crop: Tuple[float, float, float, float],
) -> Image.Image:
    w, h = full_img.size
    mid = w // 2
    top_f, bot_f, left_f, right_f = crop
    x0 = int(w * left_f);  x1 = int(w * (1.0 - right_f))
    y0 = int(h * top_f);   y1 = int(h * (1.0 - bot_f))
    x0 = max(0, min(w - 2, x0));  x1 = max(x0 + 2, min(w, x1))
    y0 = max(0, min(h - 2, y0));  y1 = max(y0 + 2, min(h, y1))
    x1 = min(x1, mid)
    if x1 <= x0 + 1:
        x0 = max(0, min(mid - 2, x0))
        x1 = max(x0 + 2, min(mid, x0 + 64))
    return full_img.crop((x0, y0, x1, y1))


# =========================
# VIDEO HELPERS
# =========================

def _video_duration_seconds(
    container: av.container.InputContainer,
    stream: av.video.stream.VideoStream,
) -> Optional[float]:
    if container.duration is not None and container.duration > 0:
        return float(container.duration) / 1_000_000.0
    if stream.duration is not None and stream.duration > 0 and stream.time_base is not None:
        return float(stream.duration * stream.time_base)
    return None


def _linspace_times(dur: float, n: int, eps: float = 0.08) -> List[float]:
    if n <= 1:
        return [0.0]
    if dur <= eps * 2:
        return [0.0] * n
    end = max(0.0, dur - eps)
    step = end / float(n - 1) if n > 1 else 0.0
    return [i * step for i in range(n)]


def motion_frame_count_from_duration(dur_s: float) -> int:
    if dur_s is None or dur_s <= 0:
        return MOTION_MIN_FRAMES
    n = int(round(dur_s * MOTION_FRAMES_PER_SEC))
    return max(MOTION_MIN_FRAMES, min(MOTION_MAX_FRAMES, n))


def sample_timeline_frames(
    video_path: str, n: int
) -> Tuple[List[Image.Image], float, List[float]]:
    container = av.open(video_path)
    stream = container.streams.video[0]
    stream.thread_type = "AUTO"

    dur = _video_duration_seconds(container, stream) or 0.0
    targets = _linspace_times(dur, n) if dur > 0.2 else [0.0] * n
    targets = [max(0.0, t) for t in targets]
    cutoff = (max(targets) if targets else 0.0) + 0.35

    frames: List[Image.Image] = []
    picked_ts: List[float] = []
    ti = 0
    last_ts_seen = 0.0

    try:
        for frame in container.decode(stream):
            if frame.pts is None or stream.time_base is None:
                continue
            ts = float(frame.pts * stream.time_base)
            if ts > last_ts_seen:
                last_ts_seen = ts

            while ti < len(targets) and ts >= targets[ti]:
                arr = frame.to_rgb().to_ndarray()
                frames.append(Image.fromarray(arr))
                picked_ts.append(ts)
                ti += 1
                if ti >= len(targets):
                    break

            if ti >= len(targets):
                break
            if cutoff > 0.0 and ts > cutoff and ti > 0:
                break
    finally:
        container.close()

    # Fallback: grab first frame and repeat
    if not frames:
        container = av.open(video_path)
        stream = container.streams.video[0]
        try:
            for frame in container.decode(stream):
                im = Image.fromarray(frame.to_rgb().to_ndarray())
                frames = [im] * n
                picked_ts = [0.0] * n
                break
        finally:
            container.close()

    while len(frames) < n:
        frames.append(frames[-1])
        picked_ts.append(picked_ts[-1] if picked_ts else 0.0)

    if dur <= 0.0 and last_ts_seen > 0.0:
        dur = last_ts_seen

    return frames[:n], float(dur), picked_ts[:n]


# =========================
# QUALITY METRICS
# =========================

def _roi_crop_np(
    gray_u8: np.ndarray,
    crop: Tuple[float, float, float, float],
) -> np.ndarray:
    h, w = gray_u8.shape
    top_f, bot_f, left_f, right_f = crop
    y0 = max(0, min(h - 2, int(h * top_f)))
    y1 = max(y0 + 2, min(h, int(h * (1.0 - bot_f))))
    x0 = max(0, min(w - 2, int(w * left_f)))
    x1 = max(x0 + 2, min(w, int(w * (1.0 - right_f))))
    return gray_u8[y0:y1, x0:x1]


def _gray_u8(img: Image.Image, size: Tuple[int, int] = (320, 180)) -> np.ndarray:
    return np.asarray(img.convert("L").resize(size), dtype=np.uint8)


def shannon_entropy_u8(arr: np.ndarray) -> float:
    hist = np.bincount(arr.reshape(-1), minlength=256).astype(np.float64)
    p = hist / (hist.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def edge_density(arr: np.ndarray) -> float:
    a = arr.astype(np.float32)
    gx = np.abs(np.diff(a, axis=1))
    gy = np.abs(np.diff(a, axis=0))
    h = min(gx.shape[0], gy.shape[0]);  w = min(gx.shape[1], gy.shape[1])
    mag = (gx[:h, :w] + gy[:h, :w])
    mag = mag / (mag.max() + 1e-6)
    return float((mag > 0.25).mean())


def laplacian_var_u8(arr: np.ndarray) -> float:
    g = arr.astype(np.float32)
    k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    h, w = g.shape
    if h < 3 or w < 3:
        return 0.0
    out = np.zeros((h - 2, w - 2), dtype=np.float32)
    for dy in range(3):
        for dx in range(3):
            out += k[dy, dx] * g[dy:dy + h - 2, dx:dx + w - 2]
    return float(out.var())


def _soft_blob_ratio(arr: np.ndarray) -> float:
    a = arr.astype(np.float32)
    if a.shape[0] < 3 or a.shape[1] < 3:
        return 0.0
    gx = np.abs(np.diff(a, axis=1))
    gy = np.abs(np.diff(a, axis=0))
    h = min(gx.shape[0], gy.shape[0]);  w = min(gx.shape[1], gy.shape[1])
    g = gx[:h, :w] + gy[:h, :w]
    return float((g < 10.0).mean()) if g.size else 0.0


def _grad_percentile(arr: np.ndarray, q: float = 75.0) -> float:
    a = arr.astype(np.float32)
    if a.shape[0] < 3 or a.shape[1] < 3:
        return 0.0
    gx = np.abs(np.diff(a, axis=1))
    gy = np.abs(np.diff(a, axis=0))
    h = min(gx.shape[0], gy.shape[0]);  w = min(gx.shape[1], gy.shape[1])
    g = gx[:h, :w] + gy[:h, :w]
    return float(np.percentile(g, q)) if g.size else 0.0


def detect_smeared_occlusion(img: Image.Image) -> Tuple[bool, Dict[str, float]]:
    arr = _gray_u8(img, (320, 180))
    roi = _roi_crop_np(arr, (0.06, 0.06, 0.04, 0.04))
    blob_ratio  = _soft_blob_ratio(roi)
    grad_p75    = _grad_percentile(roi, 75.0)
    dark_ratio  = float((roi < 25).mean())   if roi.size else 0.0
    bright_ratio = float((roi > 230).mean()) if roi.size else 0.0
    flag = (
        blob_ratio  >= SMEAR_BLOB_RATIO_T
        and grad_p75   <= SMEAR_GRAD_P75_T
        and dark_ratio  <= SMEAR_DARK_RATIO_MAX
        and bright_ratio <= SMEAR_BRIGHT_RATIO_MAX
    )
    return flag, {
        "blob_ratio":   round(blob_ratio,   4),
        "grad_p75":     round(grad_p75,     4),
        "dark_ratio":   round(dark_ratio,   4),
        "bright_ratio": round(bright_ratio, 4),
    }


def detect_foreground_obstruction(img: Image.Image) -> Tuple[bool, float]:
    arr = _gray_u8(img, (320, 180)).astype(np.float32)
    gx = np.abs(np.diff(arr, axis=1))
    gy = np.abs(np.diff(arr, axis=0))
    h = min(gx.shape[0], gy.shape[0]);  w = min(gx.shape[1], gy.shape[1])
    edges = gx[:h, :w] + gy[:h, :w]
    _, W = edges.shape
    band = edges[:, int(W * 0.35):int(W * 0.65)]
    if band.size == 0:
        return False, 0.0
    band = band / (band.max() + 1e-6)
    coverage = float((band > 0.5).mean())
    return coverage > 0.25, coverage


def classify_frame_quality(img: Image.Image) -> Tuple[bool, str, Dict[str, float]]:
    g = _gray_u8(img, (320, 180))
    roi_cov  = _roi_crop_np(g, ROI_CROP)
    ent      = shannon_entropy_u8(roi_cov)
    ed       = edge_density(roi_cov)
    roi_blur = _roi_crop_np(g, BLUR_ROI_CROP)
    lap_roi  = laplacian_var_u8(roi_blur)
    std_roi  = float(roi_blur.astype(np.float32).std())
    b        = float(g.mean())

    smear_flag, smear_dbg = detect_smeared_occlusion(img)
    base = {"brightness": b, "lap_roi": lap_roi, "std_roi": std_roi, "roi_entropy": ent, "roi_edge": ed}

    if smear_flag:
        return False, "covered", {**base, **smear_dbg}
    if lap_roi < BLUR_ROI_LAP_VAR_T and std_roi < COVERED_ROI_STD_T:
        return False, "covered", base
    if ent < COVERED_ENTROPY_T and ed < COVERED_EDGE_DENSITY_T:
        return False, "covered", base
    if b < DARK_MEAN_THRESH:
        return False, "dark", base
    if lap_roi < BLUR_ROI_LAP_VAR_T:
        return False, "blur", base
    return True, "ok", base


def decide_camera_status_from_frames(
    frames: List[Image.Image],
) -> Tuple[int, Dict[str, Any]]:
    bad_reasons: List[str] = []
    metrics: List[Dict] = []
    covered_count = dark_count = blur_count = 0
    worst_cov = 0.0

    for fr in frames:
        blocked, cov = detect_foreground_obstruction(fr)
        if cov > worst_cov:
            worst_cov = cov
        if blocked:
            return 4, {"reason": "foreground_obstruction", "coverage": cov}

        ok, reason, m = classify_frame_quality(fr)
        metrics.append(m)
        if not ok:
            bad_reasons.append(reason)
            if reason == "covered": covered_count += 1
            elif reason == "dark":  dark_count += 1
            elif reason == "blur":  blur_count += 1

    n             = max(1, len(frames))
    covered_ratio = covered_count / n
    bad_ratio     = len(bad_reasons) / n

    dbg = {
        "bad_ratio":              bad_ratio,
        "covered_ratio":          covered_ratio,
        "counts":                 {"covered": covered_count, "dark": dark_count, "blur": blur_count},
        "bad_reasons":            bad_reasons[:50],
        "metrics":                metrics[:50],
        "obstruction_max_coverage": worst_cov,
    }

    # ── Tier 1: definitely covered ────────────────────────────────────────────
    # High blob ratio + very low gradient = solid object blocking lens
    # Only triggered when enough frames are covered
    if covered_ratio >= COVERED_RATIO_T:
        blob_vals = [m.get("blob_ratio", 0) for m in metrics if "blob_ratio" in m]
        grad_vals = [m.get("grad_p75",   0) for m in metrics if "grad_p75"   in m]
        lap_vals  = [m.get("lap_roi",    0) for m in metrics if "lap_roi"    in m]

        mean_blob = sum(blob_vals) / len(blob_vals) if blob_vals else 0
        mean_grad = sum(grad_vals) / len(grad_vals) if grad_vals else 999
        mean_lap  = sum(lap_vals)  / len(lap_vals)  if lap_vals  else 999

        dbg["tier_metrics"] = {
            "mean_blob_ratio": round(mean_blob, 4),
            "mean_grad_p75":   round(mean_grad, 4),
            "mean_lap_roi":    round(mean_lap,  4),
        }

        if mean_blob >= SMEAR_BLOB_RATIO_T and mean_grad <= SMEAR_GRAD_P75_T:
            # Tier 1 — definitely covered, skip VLM
            dbg["tier"] = 1
            dbg["tier_reason"] = "high_blob_low_grad — solid obstruction"
            return 2, dbg

        if mean_lap <= LAP_ROI_COVERED_T:
            # Tier 2 — possibly covered (cream/blur/beads), send to VLM for review
            dbg["tier"] = 2
            dbg["tier_reason"] = f"low_lap_roi={mean_lap:.1f} — possible obstruction, review"
            return 5, dbg  # cam_s=5 = tier 2 review

    # ── Dark / blur ───────────────────────────────────────────────────────────
    if bad_ratio < BAD_FRAME_RATIO_THRESH:
        return 0, dbg
    if dark_count >= blur_count and dark_count > 0:
        return 2, dbg
    if blur_count > 0:
        return 3, dbg
    return 3, dbg


# =========================
# MOTION FROM FRAMES
# =========================

def _crop_np(img: Image.Image, crop: Tuple, size: Tuple = (160, 90)) -> np.ndarray:
    c = crop_frac(img, crop)
    return np.asarray(c.convert("L").resize(size), dtype=np.float32)


def _gradient_change_score(a: np.ndarray, b: np.ndarray) -> float:
    if a.size == 0 or b.size == 0:
        return 0.0
    valid = (a > 20.0) & (a < 245.0) & (b > 20.0) & (b < 245.0)
    if float(valid.mean()) < 0.20:
        return 0.0
    ax = np.diff(a, axis=1);  ay = np.diff(a, axis=0)
    bx = np.diff(b, axis=1);  by = np.diff(b, axis=0)
    vx = valid[:, 1:] & valid[:, :-1]
    vy = valid[1:, :] & valid[:-1, :]
    dx = np.abs(ax - bx)[vx]
    dy = np.abs(ay - by)[vy]
    if dx.size == 0 and dy.size == 0:
        return 0.0
    if dx.size == 0:
        return float(np.mean(dy)) / 255.0
    if dy.size == 0:
        return float(np.mean(dx)) / 255.0
    return float((np.mean(dx) + np.mean(dy)) / 2.0) / 255.0


def _pair_motion_score(a: Image.Image, b: Image.Image) -> float:
    win  = _gradient_change_score(_crop_np(a, MOTION_WINDOW_ROI),    _crop_np(b, MOTION_WINDOW_ROI))
    full_a = np.asarray(a.convert("L").resize((160, 90)), dtype=np.float32)
    full_b = np.asarray(b.convert("L").resize((160, 90)), dtype=np.float32)
    full = _gradient_change_score(full_a, full_b)
    if MOTION_USE_WINDSCREEN:
        ws = _gradient_change_score(_crop_np(a, MOTION_WINDSCREEN_ROI), _crop_np(b, MOTION_WINDSCREEN_ROI))
        return 0.60 * win + 0.25 * ws + 0.15 * full
    return 0.80 * win + 0.20 * full


def _summarize_motion_scores(scores: List[float]) -> Dict[str, float]:
    if not scores:
        return {"median": 0.0, "p75": 0.0, "max": 0.0, "active_ratio": 0.0, "high_ratio": 0.0}
    arr = np.asarray(scores, dtype=np.float32)
    return {
        "median":       float(np.median(arr)),
        "p75":          float(np.percentile(arr, 75)),
        "max":          float(arr.max()),
        "active_ratio": float((arr >= MOTION_PAIR_LOW_T).mean()),
        "high_ratio":   float((arr >= MOTION_PAIR_HIGH_T).mean()),
    }


def motion_hint_from_frames(
    frames: List[Image.Image],
) -> Tuple[int, str, float, Dict[str, Any]]:
    usable     = [fr for fr in frames if classify_frame_quality(fr)[0]]
    usable_idx = [i + 1 for i, fr in enumerate(frames) if classify_frame_quality(fr)[0]]

    if len(usable) < 2:
        return 0, "UNCERTAIN", 0.0, {"reason": "not_enough_usable_frames"}

    pair_scores = [_pair_motion_score(usable[i], usable[i + 1]) for i in range(len(usable) - 1)]
    pair_meta   = [{"pair": [usable_idx[i], usable_idx[i + 1]], "score": round(pair_scores[i], 6)}
                   for i in range(len(pair_scores))]

    stats = _summarize_motion_scores(pair_scores)

    if len(pair_scores) < MOTION_MIN_USABLE_PAIRS:
        fl = _pair_motion_score(usable[0], usable[-1])
        return 0, "UNCERTAIN", float(stats["p75"]), {
            "reason": "not_enough_usable_pairs",
            "usable_frames": usable_idx,
            "pair_scores": pair_meta[:30],
            "stats": {k: round(v, 6) for k, v in stats.items()},
            "first_last": round(fl, 6),
        }

    first_last = _pair_motion_score(usable[0], usable[-1])

    stopped = (
        stats["p75"]          <= MOTION_STOPPED_MAX_P75
        and stats["active_ratio"] <= MOTION_STOPPED_MAX_ACTIVE_RATIO
        and first_last        <= MOTION_STOPPED_MAX_FIRST_LAST
    )
    moving = (
        (stats["active_ratio"] >= MOTION_MOVING_MIN_ACTIVE_RATIO and stats["p75"] >= MOTION_MOVING_MIN_P75)
        or first_last >= MOTION_MOVING_MIN_FIRST_LAST
        or (stats["high_ratio"] >= 0.20 and stats["p75"] >= MOTION_PAIR_LOW_T)
    )

    dbg = {
        "usable_frames": usable_idx,
        "pair_scores":   pair_meta[:30],
        "stats":         {k: round(v, 6) for k, v in stats.items()},
        "first_last":    round(float(first_last), 6),
        "thresholds": {
            "pair_low":  MOTION_PAIR_LOW_T,  "pair_high": MOTION_PAIR_HIGH_T,
            "stopped_max_p75": MOTION_STOPPED_MAX_P75,
            "stopped_max_active_ratio": MOTION_STOPPED_MAX_ACTIVE_RATIO,
            "stopped_max_first_last":   MOTION_STOPPED_MAX_FIRST_LAST,
            "moving_min_active_ratio":  MOTION_MOVING_MIN_ACTIVE_RATIO,
            "moving_min_p75":           MOTION_MOVING_MIN_P75,
            "moving_min_first_last":    MOTION_MOVING_MIN_FIRST_LAST,
        },
    }
    decision_score = float(max(stats["p75"], first_last))

    if moving and not stopped:
        return 1, "MOVING",    decision_score, dbg
    if stopped and not moving:
        return 2, "STOPPED",   decision_score, dbg
    return 0,     "UNCERTAIN", decision_score, dbg


print("✓ Cell 3: Utility helpers loaded")

In [ ]:
"""
CELL 4 — VLM HELPERS
Ollama API calls, JSON parsing, output normalisation,
targeted checks, seatbelt/phone recheck, face-66 detection.
Depends on: Cell 1 (config), Cell 2 (prompts), Cell 3 (image utils)
"""

import json
import re
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import requests
from PIL import Image


# =========================
# OLLAMA CALL
# =========================

def ask_qwen_frames(
    frames: List[Image.Image],
    prompt: str,
    max_tokens: int,
    jpeg_quality: int,
) -> str:
    content: List[Dict[str, Any]] = [{"type": "text", "text": prompt}]
    for im in frames:
        data_url = pil_to_data_url(im, quality=jpeg_quality)
        content.append({"type": "image_url", "image_url": {"url": data_url}})

    payload: Dict[str, Any] = {
        "model": OLLAMA_MODEL,
        "temperature": 0.0,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": content}],
        "stop": [STOP_MARKER, "\nEND_JSON", "END_JSON"],
    }

    url = f"{OLLAMA_BASE_URL}/v1/chat/completions"
    resp = requests.post(url, json=payload, timeout=OLLAMA_TIMEOUT)
    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code} from {url}\n{resp.text[:400]}")

    data = resp.json()
    return (data["choices"][0]["message"]["content"] or "").strip()


# =========================
# JSON EXTRACTION
# =========================

def _strip_end_marker(text: str) -> str:
    return re.sub(r"\nEND_JSON\s*$", "", text.strip()).strip()


def _extract_first_complete_json(text: str) -> Dict[str, Any]:
    t = _strip_end_marker(text)
    start = t.find("{")
    if start == -1:
        raise RuntimeError(f"No JSON object found in model output:\n{t[:800]}")
    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:       esc = False
            elif ch == "\\": esc = True
            elif ch == '"':  in_str = False
            continue
        if ch == '"':   in_str = True;  continue
        if ch == "{":   depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = t[start:i + 1]
                # The VLM occasionally writes bare `u` (uncertainty token) inside list
                # fields e.g. smoking_f: [u] instead of ["u"], which is invalid JSON.
                # Sanitise before parsing by quoting any unquoted u inside list brackets.
                candidate = re.sub(r'(?<=[\[,])\s*u\s*(?=[,\]])', '"u"', candidate)
                return json.loads(candidate)
    raise RuntimeError("Model output truncated — increase MAX_TOKENS or OLLAMA_TIMEOUT.")

# =========================
# NORMALISE VLM OUTPUT
# =========================

def _safe_int(x: Any, default: int = 0) -> int:
    try:    return int(x)
    except: return default


def _safe_str(x: Any, default: str = "u") -> str:
    s = str(x).strip().lower()
    return s if s in ("y", "n", "u") else default


def _safe_yn(x: Any, default: str = "n") -> str:
    s = str(x).strip().lower()
    return s if s in ("y", "n") else default


def _frames_norm(frames_any: Any, n_frames: int) -> List[int]:
    if not isinstance(frames_any, list):
        return []
    uniq: List[int] = []
    for v in frames_any:
        try:
            iv = int(v)
            if 1 <= iv <= n_frames and iv not in uniq:
                uniq.append(iv)
        except Exception:
            pass
    return uniq


def blank_vlm_result(reason: str = "") -> Dict[str, Any]:
    return {
        "driver_vis": 0,
        "seatbelt": "u", "phone": "u", "distraction": "u",
        "fatigue": "u",  "smoking": "u",
        "eatdrink": "n", "drinking": "u", "alcohol": "u",
        "seatbelt_f": [], "phone_f": [], "distraction_f": [],
        "fatigue_f": [], "smoking_f": [], "eatdrink_f": [],
        "why": reason[:120],
    }


def normalize_vlm(obj: Dict[str, Any], n_frames: int) -> Dict[str, Any]:
    driver_vis  = _safe_int(obj.get("driver_vis", 0), 0)
    driver_vis  = driver_vis if driver_vis in (0, 1, 2) else 0

    seatbelt    = _safe_str(obj.get("seatbelt",    "u"), "u")
    phone       = _safe_str(obj.get("phone",       "u"), "u")
    distraction = _safe_str(obj.get("distraction", "u"), "u")
    fatigue     = _safe_str(obj.get("fatigue",     "u"), "u")
    smoking     = _safe_str(obj.get("smoking",     "u"), "u")
    eatdrink    = _safe_yn( obj.get("eatdrink",    "n"), "n")
    drinking    = _safe_str(obj.get("drinking",    "u"), "u")
    alcohol     = _safe_str(obj.get("alcohol",     "u"), "u")

    seatbelt_f    = _frames_norm(obj.get("seatbelt_f",    []), n_frames)
    phone_f       = _frames_norm(obj.get("phone_f",       []), n_frames)
    distraction_f = _frames_norm(obj.get("distraction_f", []), n_frames)
    fatigue_f     = _frames_norm(obj.get("fatigue_f",     []), n_frames)
    smoking_f     = _frames_norm(obj.get("smoking_f",     []), n_frames)
    eatdrink_f    = _frames_norm(obj.get("eatdrink_f",    []), n_frames)
    why           = str(obj.get("why", "")).strip()[:120]

    # Hard constraint: driver not visible
    if driver_vis == 2:
        seatbelt = phone = distraction = fatigue = smoking = drinking = alcohol = "u"
        eatdrink = "n"
        seatbelt_f = phone_f = distraction_f = fatigue_f = smoking_f = eatdrink_f = []

    # Clear frame lists for uncertain fields
    if seatbelt    == "u": seatbelt_f    = []
    if phone       == "u": phone_f       = []
    if distraction == "u": distraction_f = []
    if fatigue     == "u": fatigue_f     = []
    if smoking     == "u": smoking_f     = []
    if eatdrink    == "n": eatdrink_f    = []

    return {
        "driver_vis": driver_vis,
        "seatbelt": seatbelt, "phone": phone, "distraction": distraction,
        "fatigue": fatigue,   "smoking": smoking,
        "eatdrink": eatdrink, "drinking": drinking, "alcohol": alcohol,
        "seatbelt_f": seatbelt_f, "phone_f": phone_f, "distraction_f": distraction_f,
        "fatigue_f": fatigue_f,   "smoking_f": smoking_f, "eatdrink_f": eatdrink_f,
        "why": why,
    }


# =========================
# TARGETED VLM CHECKS
# =========================

def label_to_target(event_description: str) -> Optional[str]:
    mapping = {
        "v_cam_covered":  "cam_covered",
        "v_distraction":  "distraction",
        "v_eye_closed":   "fatigue",
        "v_fatigue":      "fatigue",
        "v_yawn":         "fatigue",
        "v_phone":        "phone",
        "v_smoke":        "smoking",
        "SEATBELT_D_OFF": "seatbelt",
    }
    return mapping.get(event_description)


def run_targeted_check(frames_left: List[Image.Image], target: str) -> Dict[str, Any]:
    prompt_map = {
        "smoking":    SMOKING_ONLY_PROMPT,
        "phone":      PHONE_ONLY_PROMPT,
        "seatbelt":   SEATBELT_ONLY_PROMPT,
        "distraction": DISTRACTION_ONLY_PROMPT,
        "fatigue":    FATIGUE_ONLY_PROMPT,
    }
    if target not in prompt_map:
        raise ValueError(f"Unsupported target: {target!r}")

    raw = ask_qwen_frames(frames_left, prompt_map[target], max_tokens=220, jpeg_quality=JPEG_QUALITY)
    obj = _extract_first_complete_json(raw)

    out = blank_vlm_result(reason=f"targeted_{target}")
    out["driver_vis"] = _safe_int(obj.get("driver_vis", 0), 0)
    out["why"]        = str(obj.get("why", "")).strip()[:120]

    field_map = {
        "smoking":    ("smoking",    "smoking_f"),
        "phone":      ("phone",      "phone_f"),
        "seatbelt":   ("seatbelt",   "seatbelt_f"),
        "distraction": ("distraction", "distraction_f"),
        "fatigue":    ("fatigue",    "fatigue_f"),
    }
    val_key, frames_key = field_map[target]
    out[val_key]    = _safe_str(obj.get(val_key, "u"), "u")
    out[frames_key] = _frames_norm(obj.get(frames_key, []), NUM_FRAMES)

    # Hard constraint: driver not visible
    if out["driver_vis"] == 2:
        for k in ("seatbelt", "phone", "distraction", "fatigue", "smoking", "drinking", "alcohol"):
            out[k] = "u"
        out["eatdrink"] = "n"
        for k in ("seatbelt_f", "phone_f", "distraction_f", "fatigue_f", "smoking_f", "eatdrink_f"):
            out[k] = []

    return out


# =========================
# RECHECK: SEATBELT + PHONE
# =========================

def recheck_seatbelt_phone(
    full_frames: List[Image.Image],
    clip_dir: str,
) -> Dict[str, Any]:
    cropped: List[Image.Image] = []
    crop_frame_map: List[int] = []

    for frame_idx, fr_full in enumerate(full_frames, start=1):
        for crop in RECHECK_ROI_CROPS:
            c = crop_frac_from_full_then_clip_to_left(fr_full, crop)
            c = resize_image_keep_aspect(c, RECHECK_FRAME_MAX)
            cropped.append(c)
            crop_frame_map.append(frame_idx)

    raw = ask_qwen_frames(
        cropped,
        RECHECK_SEATBELT_PHONE_PROMPT,
        max_tokens=RECHECK_MAX_TOKENS,
        jpeg_quality=RECHECK_JPEG_QUALITY,
    )
    obj = _extract_first_complete_json(raw)

    sb  = _safe_str(obj.get("seatbelt", "u"), "u")
    ph  = _safe_str(obj.get("phone",    "u"), "u")
    why = str(obj.get("why", "")).strip()[:120]

    def _map_support(v_raw: Any) -> List[int]:
        uniq: List[int] = []
        if isinstance(v_raw, list):
            for v in v_raw:
                try:
                    iv = int(v)
                except Exception:
                    continue
                # Map from crop-index back to original frame number
                orig = crop_frame_map[iv - 1] if 1 <= iv <= len(crop_frame_map) else iv
                if 1 <= orig <= NUM_FRAMES and orig not in uniq:
                    uniq.append(orig)
        return uniq

    sb_f = _map_support(obj.get("seatbelt_f", []))
    ph_f = _map_support(obj.get("phone_f",    []))
    if sb == "u": sb_f = []
    if ph == "u": ph_f = []

    return {"seatbelt": sb, "phone": ph, "seatbelt_f": sb_f, "phone_f": ph_f, "why": why}


# =========================
# FACE OBSTRUCTION 66
# =========================

def detect_potential_face_obstruction_66(
    frames_left: List[Image.Image],
) -> Tuple[bool, Dict[str, Any]]:
    if not FACE66_ENABLED:
        return False, {"reason": "disabled"}

    try:
        import cv2
    except Exception as e:
        return False, {"reason": f"cv2_unavailable:{str(e)[:80]}"}

    try:
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        face_cascade = cv2.CascadeClassifier(cascade_path)
        if face_cascade.empty():
            return False, {"reason": "cascade_empty"}
    except Exception as e:
        return False, {"reason": f"cascade_error:{str(e)[:80]}"}

    usable = noface = lowinfo = 0
    per_frame: List[Dict] = []

    for idx, fr in enumerate(frames_left, start=1):
        ok, _, _ = classify_frame_quality(fr)
        if not ok:
            per_frame.append({"f": idx, "usable": 0})
            continue

        usable += 1
        arr  = np.asarray(fr.resize((320, 180)).convert("L"), dtype=np.uint8)
        h, w = arr.shape
        roi  = arr[int(h * 0.05):int(h * 0.78), int(w * 0.05):int(w * 0.85)]
        faces = face_cascade.detectMultiScale(roi, scaleFactor=1.1, minNeighbors=4, minSize=(28, 28))

        face_zone = arr[int(h * 0.08):int(h * 0.55), int(w * 0.08):int(w * 0.60)]
        zone_std  = float(face_zone.astype(np.float32).std()) if face_zone.size else 0.0
        zone_ent  = shannon_entropy_u8(face_zone) if face_zone.size else 0.0
        li_here   = zone_std < FACE66_ZONE_STD_T and zone_ent < FACE66_ZONE_ENT_T

        if len(faces) == 0: noface  += 1
        if li_here:          lowinfo += 1

        per_frame.append({
            "f": idx, "usable": 1, "faces": int(len(faces)),
            "zone_std": round(zone_std, 2), "zone_ent": round(zone_ent, 2),
            "lowinfo": int(li_here),
        })

    if usable < FACE66_MIN_USABLE_FRAMES:
        return False, {"reason": "not_enough_usable_frames", "usable": usable, "per_frame": per_frame[:50]}

    noface_ratio  = noface  / float(usable)
    lowinfo_ratio = lowinfo / float(usable)
    flag = noface_ratio >= FACE66_NOFACE_RATIO_T and lowinfo_ratio >= FACE66_LOWINFO_RATIO_T

    return flag, {
        "usable": usable, "noface": noface, "lowinfo": lowinfo,
        "noface_ratio":  round(noface_ratio,  3),
        "lowinfo_ratio": round(lowinfo_ratio, 3),
        "per_frame": per_frame[:50],
    }


print("✓ Cell 4: VLM helpers loaded")

In [ ]:
"""
CELL 5 — DECISION ENGINE & REVIEW SCORING
decide_events, review scoring, labelled agreement.
Depends on: Cell 1 (EVENT_CODE, LABELLED_EVENTS, REVIEW_ON_SEATBELT_U_WHEN_MOVING)
"""

from typing import Dict, List, Tuple


def decide_events(
    cam_s: int,
    vs: int,
    cam_o: int,
    driver_vis: int,
    seatbelt: str,
    phone: str,
    distraction: str,
    fatigue: str,
    smoking: str,
) -> Tuple[List[int], List[int]]:
    """
    Returns (det_codes, rev_codes).
    det  = confirmed detections
    rev  = flagged for human review
    """
    det: List[int] = []
    rev: List[int] = []

    # Camera bad — no behaviour analysis possible
    if cam_s in (2, 3):
        det.append(EVENT_CODE["CAMERA_COVERED"])
        return sorted(set(det)), sorted(set(rev))

    if cam_s == 4:
        rev.append(EVENT_CODE["OBSTRUCTION_VIEW"])
        return sorted(set(det)), sorted(set(rev))

    # Only cabin cameras are analysed
    if cam_o != 2:
        return sorted(set(det)), sorted(set(rev))

    # Driver must be clearly visible
    if driver_vis != 1:
        return sorted(set(det)), sorted(set(rev))

    # --- Seatbelt ---
    if seatbelt == "n":
        (det if vs == 1 else rev).append(EVENT_CODE["SEATBELT_OFF_DRIVER"])
    elif seatbelt == "u" and vs == 1 and REVIEW_ON_SEATBELT_U_WHEN_MOVING:
        rev.append(EVENT_CODE["SEATBELT_OFF_DRIVER"])

    # --- Phone ---
    if phone == "y":
        (det if vs == 1 else rev).append(EVENT_CODE["PHONE_DRIVER"])

    # --- Distraction ---
    if distraction == "y":
        (det if vs == 1 else rev).append(EVENT_CODE["DISTRACTION"])

    # --- Fatigue ---
    if fatigue == "y":
        (det if vs == 1 else rev).append(EVENT_CODE["FATIGUE_DRIVER"])

    # --- Smoking ---
    if smoking == "y":
        (det if vs == 1 else rev).append(EVENT_CODE["SMOKING_DRIVER"])

    return sorted(set(det)), sorted(set(rev))


# =========================
# REVIEW SCORING
# =========================

def _review_score_seatbelt(seatbelt: str, vs: int) -> str:
    if seatbelt == "n" and vs == 1:
        return "high"
    if (seatbelt == "u" and vs == 1) or (seatbelt == "n" and vs != 1):
        return "medium"
    return "low"


def _review_score_phone(phone: str, vs: int) -> str:
    if phone == "y" and vs == 1:
        return "high"
    if (phone == "y" and vs != 1) or (phone == "u" and vs == 1):
        return "medium"
    return "low"


def _review_score_drinking(drinking: str, alcohol: str) -> str:
    if drinking == "y" and alcohol == "y":
        return "high"
    if drinking == "y":
        return "medium"
    return "low"


# =========================
# LABELLED AGREEMENT
# =========================

def labelled_agreement(
    event_description: str,
    labelled_checked: bool,
    cam_s: int,
    seatbelt: str,
    phone: str,
    distraction: str,
    fatigue: str,
    smoking: str,
    vs: int,
) -> str:
    if not labelled_checked:
        return "na"

    if event_description == "v_cam_covered":
        return "y" if cam_s in (2, 3) else "n"
    if event_description == "v_phone":
        return "y" if phone == "y" else "n"
    if event_description == "v_distraction":
        return "y" if distraction == "y" else "n"
    if event_description in ("v_eye_closed", "v_fatigue", "v_yawn"):
        return "y" if fatigue == "y" else "n"
    if event_description == "v_smoke":
        return "y" if smoking == "y" else "n"
    if event_description == "SEATBELT_D_OFF":
        return "y" if seatbelt == "n" and vs == 1 else "n"
    return "na"


print("✓ Cell 5: Decision engine loaded")

In [ ]:
"""
CELL 6 — INCREMENTAL RUN HELPERS
File I/O, source-meta management, alignment audit, result validation,
pending-job filtering, and summary CSV writer.
Depends on: Cell 1 (OUTPUT_BASE_DIR, RESULT_FILENAME, ERROR_FILENAME, SOURCE_META_FILENAME)
"""

import hashlib
import json
import os
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


# =========================
# NORMALISATION
# =========================

def _normalize_url(u: Any) -> str:
    try:
        if pd.isna(u):
            return ""
    except Exception:
        pass
    return str(u).strip().rstrip("/")


def _normalize_event_description(v: Any) -> str:
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v).strip()


def _safe_int_any(v: Any, default: int = 0) -> int:
    try:    return int(v)
    except: return default


# =========================
# PATH HELPERS
# =========================

def _clip_dir(clip_no: int, base: str = OUTPUT_BASE_DIR) -> str:
    return os.path.join(base, f"clip_{clip_no:03d}")


# =========================
# SOURCE HASH + META
# =========================

def _build_source_hash(url: str) -> str:
    return hashlib.sha256(_normalize_url(url).encode("utf-8")).hexdigest()[:16]


def _build_source_meta(item: dict, clip_no: int) -> dict:
    norm_url = _normalize_url(item.get("url", ""))
    try:    camera_int = int(item.get("camera", 0))
    except: camera_int = 0
    return {
        "clip":              int(clip_no),
        "source_hash":       _build_source_hash(norm_url),
        "camera":            camera_int,
        "event_description": _normalize_event_description(item.get("event_description", "")),
    }


# =========================
# JSON I/O
# =========================

def _write_json(path: str, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def _read_json(path: str) -> dict:
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


def _write_source_meta(clip_dir: str, item: dict, clip_no: int) -> dict:
    meta = _build_source_meta(item=item, clip_no=clip_no)
    _write_json(os.path.join(clip_dir, SOURCE_META_FILENAME), meta)
    return meta


def _read_source_meta(clip_dir: str) -> dict:
    return _read_json(os.path.join(clip_dir, SOURCE_META_FILENAME))


# =========================
# RESULT VALIDATION
# =========================

def _extract_event_codes_list(v: Any) -> List[int]:
    if not isinstance(v, list):
        return []
    out = []
    for item in v:
        if isinstance(item, dict) and "e" in item:
            try: out.append(int(item["e"]))
            except: pass
        else:
            try: out.append(int(item))
            except: pass
    return out


def _is_valid_result_json(result: dict) -> Tuple[bool, str]:
    if not isinstance(result, dict) or not result:
        return False, "result_not_dict_or_empty"

    for k in ("cam", "scene", "meta", "debug"):
        if k not in result or not isinstance(result[k], dict):
            return False, f"missing_or_invalid_top_key:{k}"

    cam   = result["cam"]
    scene = result["scene"]
    meta  = result["meta"]

    for k, parent, label in (("o", cam, "cam"), ("s", cam, "cam"), ("vs", scene, "scene")):
        if k not in parent:
            return False, f"{label}_missing_{k}"
        try: int(parent[k])
        except: return False, f"{label}_{k}_not_int"

    for k in ("event_description", "labelled_event", "labelled_checked", "labelled_agreed"):
        if k not in meta:
            return False, f"meta_missing_{k}"

    for k in ("det", "rev"):
        if k in result:
            if not isinstance(result[k], list):
                return False, f"{k}_not_list"
            for item in result[k]:
                if not isinstance(item, dict) or "e" not in item:
                    return False, f"{k}_bad_item"

    if "status" in result and not isinstance(result["status"], str):
        return False, "status_not_string"
    if "timing" in result and not isinstance(result["timing"], dict):
        return False, "timing_not_dict"

    return True, "ok"


def _read_validated_result(result_path: str) -> Tuple[dict, bool, str]:
    result = _read_json(result_path)
    is_valid, reason = _is_valid_result_json(result)
    return result, is_valid, reason


def _is_clip_successfully_processed(
    clip_no: int,
    base: str = OUTPUT_BASE_DIR,
) -> Tuple[bool, str]:
    result_path = os.path.join(_clip_dir(clip_no, base), RESULT_FILENAME)
    result, is_valid, reason = _read_validated_result(result_path)
    if not is_valid:
        return False, reason
    status = str(result.get("status", "OK")).strip().upper()
    if status not in {"OK", "SUCCESS"}:
        return False, f"status_not_success:{status}"
    return True, "ok"


# =========================
# ALIGNMENT AUDIT
# =========================

# =========================
# HASH → CLIP DIR INDEX
# =========================

def _build_hash_to_clip_map(base: str = OUTPUT_BASE_DIR) -> Dict[str, int]:
    """
    Scan all existing clip dirs and return {source_hash: clip_no}
    by reading their source_meta.json files.
    """
    mapping: Dict[str, int] = {}
    if not os.path.isdir(base):
        return mapping
    for name in os.listdir(base):
        m = re.fullmatch(r"clip_(\d+)", name)
        if not m:
            continue
        clip_no = int(m.group(1))
        meta = _read_source_meta(os.path.join(base, name))
        h = str(meta.get("source_hash", "")).strip()
        if h:
            mapping[h] = clip_no
    return mapping


def _next_clip_no(base: str = OUTPUT_BASE_DIR) -> int:
    """Return the next available clip number (max existing + 1, or 1)."""
    if not os.path.isdir(base):
        return 1
    existing = []
    for name in os.listdir(base):
        m = re.fullmatch(r"clip_(\d+)", name)
        if m:
            existing.append(int(m.group(1)))
    return max(existing, default=0) + 1


# =========================
# INCREMENTAL FILTER
# =========================

def _prepare_video_urls_for_incremental_run(
    VIDEO_URLS: list,
    base: str = OUTPUT_BASE_DIR,
) -> dict:
    # --- Deduplicate and normalise input ---
    seen_urls: set = set()
    filtered: List[dict] = []
    for item in VIDEO_URLS:
        if not isinstance(item, dict):
            continue
        norm_url = _normalize_url(item.get("url", ""))
        if not norm_url or norm_url in seen_urls:
            continue
        seen_urls.add(norm_url)
        filtered.append({
            **item,
            "url":               norm_url,
            "event_description": _normalize_event_description(item.get("event_description", "")),
        })

    # --- Load what is already on disk (keyed by hash) ---
    hash_to_clip = _build_hash_to_clip_map(base)
    next_clip_no = _next_clip_no(base)

    processed_clip_numbers: List[int] = []
    invalid_existing: List[dict] = []
    pending_jobs: List[dict] = []

    # Assign a stable clip_no to every item in the current list
    # Re-use the existing clip_no if the hash is already on disk,
    # otherwise allocate the next available number.
    all_video_urls_with_clip: List[dict] = []

    for item in filtered:
        norm_url = item["url"]
        h = _build_source_hash(norm_url)

        if h in hash_to_clip:
            # Already exists on disk — check if it completed successfully
            clip_no = hash_to_clip[h]
            ok, reason = _is_clip_successfully_processed(clip_no, base)
            enriched = {**item, "_clip_no": clip_no, "_hash": h}
            all_video_urls_with_clip.append(enriched)
            if ok:
                processed_clip_numbers.append(clip_no)
            else:
                invalid_existing.append({"clip": clip_no, "reason": reason})
                pending_jobs.append({"clip_no": clip_no, "item": item})
        else:
            # Brand new — allocate next clip number
            clip_no = next_clip_no
            next_clip_no += 1
            enriched = {**item, "_clip_no": clip_no, "_hash": h}
            all_video_urls_with_clip.append(enriched)
            pending_jobs.append({"clip_no": clip_no, "item": item})

    # ALL_VIDEO_URLS strips the internal _clip_no/_hash keys before returning
    all_video_urls_clean = [
        {k: v for k, v in item.items() if not k.startswith("_")}
        for item in all_video_urls_with_clip
    ]

    return {
        "ALL_VIDEO_URLS":         all_video_urls_clean,
        "ALL_VIDEO_URLS_INTERNAL": all_video_urls_with_clip,  # includes _clip_no/_hash
        "PENDING_JOBS":           pending_jobs,
        "PROCESSED_CLIP_NUMBERS": processed_clip_numbers,
        "INVALID_EXISTING_CLIPS": invalid_existing,
    }


# =========================
# SUMMARY CSV
# =========================

@dataclass
class SummaryRow:
    clip: int
    url: str
    camera: int
    event_description: str
    labelled_event: str
    labelled_checked: str
    labelled_agreed: str
    status: str
    cam_o: int
    cam_s: int
    vs: int
    det: str
    rev: str
    fatigue: str
    smoking: str
    eatdrink: str
    review_seatbelt: str
    review_phone: str
    review_drinking: str
    t_total: float
    t_download: float
    t_frames: float
    t_llm: float


def _codes_str(codes: List[int]) -> str:
    return ",".join(str(c) for c in sorted(set(codes)))


def write_summary_csv(
    all_video_urls_internal: List[dict],
    base: str,
    path: str,
) -> None:
    """
    Rebuilds the summary CSV from scratch by reading every clip's result.json.
    all_video_urls_internal must be the ALL_VIDEO_URLS_INTERNAL list (contains _clip_no).
    """
    fields = [
        "clip", "camera", "event_description", "source_hash",
        "source_meta_ok", "result_valid", "result_valid_reason",
        "status", "cam_o", "cam_s", "vs",
        "det_codes", "rev_codes",
        "fatigue", "smoking", "eatdrink",
        "review_seatbelt", "review_phone", "review_drinking",
        "t_total_s", "t_download_s", "t_frames_s", "t_llm_s",
        "error", "labelled_checked",
"labelled_agreed",
    ]

    def _s(v, d=""): return "" if v is None else str(v).strip() or d
    def _i(v, d=0):
        try:   return int(v)
        except: return d
    def _f(v, d=""):
        try:   return f"{float(v):.2f}"
        except: return d

    records = []
    for item in all_video_urls_internal:
        i      = item["_clip_no"]
        clip_d = _clip_dir(i, base)
        result_path = os.path.join(clip_d, RESULT_FILENAME)
        error_path  = os.path.join(clip_d, ERROR_FILENAME)

        expected     = _build_source_meta(item=item, clip_no=i)
        source_meta  = _read_source_meta(clip_d)
        source_meta_ok = (
            bool(source_meta)
            and _safe_int_any(source_meta.get("clip", -1)) == i
            and str(source_meta.get("source_hash", "")) == expected["source_hash"]
            and _safe_int_any(source_meta.get("camera", 0)) == expected["camera"]
            and _normalize_event_description(source_meta.get("event_description", "")) == expected["event_description"]
        )

        result, result_valid, result_valid_reason = _read_validated_result(result_path)

        err_msg = ""
        if os.path.exists(error_path):
            try:
                with open(error_path, "r", encoding="utf-8") as f:
                    err_msg = f.read().strip()
            except Exception:
                pass

        cam    = result.get("cam",   {}) if isinstance(result.get("cam"),   dict) else {}
        scene  = result.get("scene", {}) if isinstance(result.get("scene"), dict) else {}
        debug  = result.get("debug", {}) if isinstance(result.get("debug"), dict) else {}
        vlm    = debug.get("vlm",    {}) if isinstance(debug.get("vlm"),    dict) else {}
        timing = result.get("timing",{}) if isinstance(result.get("timing"),dict) else {}

        if result_valid:
            status = _s(result.get("status", "OK"), "OK")
        elif err_msg:
            status = "FAILED"
        elif os.path.isdir(clip_d):
            status = "INCOMPLETE"
        else:
            status = ""

        records.append({
            "clip":                 i,
            "camera":               expected["camera"],
            "event_description":    expected["event_description"],
            "source_hash":          expected["source_hash"],
            "source_meta_ok":       "y" if source_meta_ok else "n",
            "result_valid":         "y" if result_valid   else "n",
            "result_valid_reason":  result_valid_reason,
            "status":               status,
            "cam_o":                _i(cam.get("o", 0)),
            "cam_s":                _i(cam.get("s", 0)),
            "vs":                   _i(scene.get("vs", 0)),
            "det_codes":            ",".join(str(x) for x in _extract_event_codes_list(result.get("det", []))),
            "rev_codes":            ",".join(str(x) for x in _extract_event_codes_list(result.get("rev", []))),
            "fatigue":              _s(vlm.get("fatigue",  "")),
            "smoking":              _s(vlm.get("smoking",  "")),
            "eatdrink":             _s(vlm.get("eatdrink", "")),
            "review_seatbelt":      _s(vlm.get("review_scores", {}).get("seatbelt",  "") if isinstance(vlm.get("review_scores"), dict) else ""),
            "review_phone":         _s(vlm.get("review_scores", {}).get("phone",     "") if isinstance(vlm.get("review_scores"), dict) else ""),
            "review_drinking":      _s(vlm.get("review_scores", {}).get("drinking",  "") if isinstance(vlm.get("review_scores"), dict) else ""),
            "t_total_s":            _f(timing.get("total_s",    "")),
            "t_download_s":         _f(timing.get("download_s", "")),
            "t_frames_s":           _f(timing.get("frames_s",   "")),
            "t_llm_s":              _f(timing.get("llm_s",      "")),
            "error":                err_msg,
            "labelled_checked": _s(result.get("meta", {}).get("labelled_checked", "")),
            "labelled_agreed":  _s(result.get("meta", {}).get("labelled_agreed",  "")),
        })

    pd.DataFrame(records, columns=fields).to_csv(path, index=False, encoding="utf-8")
    print(f"  Summary CSV written: {os.path.abspath(path)}  ({len(records)} rows)")


print("✓ Cell 6: Incremental run helpers loaded")

In [ ]:
### Final build for raw_df csv# =============================================================================
# SAVE HASH BRIDGE
# =============================================================================
# Builds source_hash from video_df URLs and saves a lookup table mapping
# source_hash → vehicle_id + terminal_event_id.
# URLs are never written to disk — only the hash is exported.
# Run this once after Cell 6, before sharing outputs/.
# =============================================================================

hash_bridge = video_df[["url", "vehicle_id", "terminal_event_id"]].copy()

# Debug: what does terminal_event_id look like before any processing?
print("=== BEFORE PROCESSING ===")
print(f"dtype                : {hash_bridge['terminal_event_id'].dtype}")
print(f"notna count          : {hash_bridge['terminal_event_id'].notna().sum()}")
print(f"sample raw values    :")
print(hash_bridge["terminal_event_id"].dropna().head(10).tolist())

hash_bridge["source_hash"] = hash_bridge["url"].apply(_build_source_hash)
hash_bridge = hash_bridge.drop(columns=["url"])
hash_bridge["vehicle_id"] = pd.to_numeric(hash_bridge["vehicle_id"], errors="coerce")

# Keep terminal_event_id as string to avoid float64 precision loss on 64-bit integers
hash_bridge["terminal_event_id"] = hash_bridge["terminal_event_id"].astype(str).str.strip()

# Replace "nan" / "None" strings back to None
hash_bridge["terminal_event_id"] = hash_bridge["terminal_event_id"].where(
    ~hash_bridge["terminal_event_id"].isin(["nan", "None", "NaT", ""]),
    other=None
)

print("\n=== AFTER PROCESSING ===")
print(f"notna count          : {hash_bridge['terminal_event_id'].notna().sum()}")
print(f"sample values        :")
print(hash_bridge["terminal_event_id"].dropna().head(10).tolist())

hash_bridge = hash_bridge.drop_duplicates(subset=["source_hash"]).reset_index(drop=True)

# Save — terminal_event_id stays as string in CSV
hash_bridge.to_csv(os.path.join("hash_bridge.csv"), index=False)

print(f"\n=== SAVED ===")
print(f"Rows saved              : {len(hash_bridge)}")
print(f"vehicle_id notna        : {hash_bridge['vehicle_id'].notna().sum()}")
print(f"terminal_event_id notna : {hash_bridge['terminal_event_id'].notna().sum()}")
print(f"source_hash notna       : {hash_bridge['source_hash'].notna().sum()}")

# Verify by reloading — MUST use dtype=str to prevent float64 precision loss
verify = pd.read_csv(
    os.path.join("hash_bridge.csv"),
    dtype={"terminal_event_id": str}
)
print(f"\n=== VERIFY RELOAD ===")
print(f"dtype after reload      : {verify['terminal_event_id'].dtype}")
print(f"notna after reload      : {verify['terminal_event_id'].notna().sum()}")
print(f"sample after reload     :")
print(verify["terminal_event_id"].dropna().head(10).tolist())

# Confirm no precision loss — compare raw vs reloaded
print(f"\n=== PRECISION CHECK ===")
raw_sample    = hash_bridge["terminal_event_id"].dropna().head(5).tolist()
reload_sample = verify["terminal_event_id"].dropna().head(5).tolist()
for r, v in zip(raw_sample, reload_sample):
    match = "✓" if str(r) == str(v) else "✗ MISMATCH"
    print(f"  {r}  →  {v}  {match}")

## Change amount of clips here

In [ ]:
"""
CELL 7 — INPUT & INCREMENTAL STATE
Define VIDEO_URLS here (edit as needed), then run this cell to
compute ALL_VIDEO_URLS, PENDING_JOBS, etc.

Depends on: Cells 1–6
"""

# ------------------------------------------------------------------
# EDIT THIS BLOCK — define your input however you like.
# The result must be a list of dicts with keys:
#   url               (str)   required
#   camera            (int)   2 = cabin camera, anything else = ignored
#   event_description (str)   optional, used for labelled agreement
# ------------------------------------------------------------------

# Example A: from a DataFrame (most common usage)
# VIDEO_URLS = (
#     video_df[["url", "camera", "event_description"]]
#     .dropna(subset=["url"])
#     .astype({"url": str})
#     .head(10)
#     .to_dict("records")
# )

# Example B: hard-coded list
# VIDEO_URLS = [
#     {"url": "https://example.com/clip1.mp4", "camera": 2, "event_description": "v_phone"},
#     {"url": "https://example.com/clip2.mp4", "camera": 2, "event_description": "SEATBELT_D_OFF"},
# ]

# ------------------------------------------------------------------
# Replace the line below with your actual VIDEO_URLS definition
# ------------------------------------------------------------------
VIDEO_URLS = (
    video_df[["url", "camera", "event_description"]]
    .dropna(subset=["url"])
    .astype({"url": str})
    .head(4500)
    .to_dict("records")
)

# ------------------------------------------------------------------
# Run incremental filter — do not edit below this line
# ------------------------------------------------------------------
_state = _prepare_video_urls_for_incremental_run(
    VIDEO_URLS=VIDEO_URLS,
    base=OUTPUT_BASE_DIR,
)

ALL_VIDEO_URLS          = _state["ALL_VIDEO_URLS"]
ALL_VIDEO_URLS_INTERNAL = _state["ALL_VIDEO_URLS_INTERNAL"]
PENDING_JOBS            = _state["PENDING_JOBS"]
PROCESSED_CLIP_NUMBERS  = _state["PROCESSED_CLIP_NUMBERS"]
INVALID_EXISTING_CLIPS  = _state["INVALID_EXISTING_CLIPS"]

print(f"Total videos in master list : {len(ALL_VIDEO_URLS)}")
print(f"Already successfully done   : {len(PROCESSED_CLIP_NUMBERS)}")
print(f"Invalid / incomplete on disk: {len(INVALID_EXISTING_CLIPS)}")
if INVALID_EXISTING_CLIPS:
    for row in INVALID_EXISTING_CLIPS[:10]:
        print(f"  clip_{row['clip']:03d}: {row['reason']}")
print(f"Pending this run            : {len(PENDING_JOBS)}")
if PENDING_JOBS:
    print("  Pending clip numbers:", [x["clip_no"] for x in PENDING_JOBS[:20]])

In [ ]:
# To monitor order of clips coming up. Mainly used to test if a failed vlm clip from earlier was going to rerun.
print([x["clip_no"] for x in PENDING_JOBS if x["clip_no"] <= 720])

In [ ]:
"""
CELL 8 — MAIN PROCESSING LOOP
Iterates over PENDING_JOBS and processes each clip.
Depends on: Cells 1–7 (all helpers + PENDING_JOBS / ALL_VIDEO_URLS)

Camera quality tiers (cam_s values):
  0 = camera OK         — VLM runs normally
  2 = tier 1 covered    — solid obstruction confirmed, skip VLM, det=[6]
  3 = blur              — skip VLM
  4 = foreground obj    — skip VLM, rev=[4]
  5 = tier 2 review     — possible covering (cream/beads/blur), VLM runs, rev=[67]
"""

import os
import csv
import time
from typing import Any, Dict, List, Optional

# -----------------------------------------------------------------------
# Safety guard: make sure earlier cells have been run
# -----------------------------------------------------------------------
for _required in ("PENDING_JOBS", "ALL_VIDEO_URLS", "OUTPUT_BASE_DIR",
                  "ask_qwen_frames", "decide_events", "write_summary_csv"):
    if _required not in dir():
        raise NameError(
            f"'{_required}' is not defined. "
            "Please run Cells 1–7 before running this cell."
        )

# -----------------------------------------------------------------------
# Pre-build the full VLM prompt once
# -----------------------------------------------------------------------
full_prompt = build_prompt()

# -----------------------------------------------------------------------
# MAIN LOOP
# -----------------------------------------------------------------------
print("=" * 80)
print(f"Model      : {OLLAMA_MODEL}")
print(f"Ollama URL : {OLLAMA_BASE_URL}")
print(f"Pending    : {len(PENDING_JOBS)} clip(s)")
print(f"Frames/clip: {NUM_FRAMES}")
print("=" * 80)

if not PENDING_JOBS:
    print("Nothing to do — all clips already successfully processed.")
else:
    for run_idx, job in enumerate(PENDING_JOBS, start=1):
        clip_no           = int(job["clip_no"])
        item              = job["item"]
        url               = str(item.get("url", "")).strip()
        event_description = _normalize_event_description(item.get("event_description", ""))
        labelled_event    = "y" if event_description in LABELLED_EVENTS else "n"
        labelled_checked  = "n"
        labelled_agreed   = "na"

        try:    camera_int = int(item.get("camera", 0))
        except: camera_int = 0

        clip_d = _clip_dir(clip_no, OUTPUT_BASE_DIR)
        os.makedirs(clip_d, exist_ok=True)

        # Write source meta and clear any old error file
        _write_source_meta(clip_dir=clip_d, item=item, clip_no=clip_no)
        err_path = os.path.join(clip_d, ERROR_FILENAME)
        if os.path.exists(err_path):
            try: os.remove(err_path)
            except: pass

        t0_total = time.perf_counter()
        print(f"\n[{run_idx}/{len(PENDING_JOBS)}] clip {clip_no:03d}  camera={camera_int}  event={event_description!r}")
        print(f"  URL: {url}")

        # Initialise state for this clip
        local_path: Optional[str] = None
        t_download = t_frames = t_llm = 0.0
        cam_s = vs = 0
        cam_o = 2 if camera_int == 2 else 1
        det_codes: List[int] = []
        rev_codes: List[int] = []
        driver_vis = 0
        seatbelt = phone = distraction = fatigue = smoking = "u"
        eatdrink = "n"
        drinking = alcohol = "u"
        status = "OK"
        review_scores = {"seatbelt": "low", "phone": "low", "drinking": "low"}
        review_items: List[str] = []
        vlm_norm: Dict[str, Any] = blank_vlm_result(reason="not_run")
        potential66   = False
        face66_dbg: Dict[str, Any] = {"reason": "not_run"}
        sel_dbg = drop_dbg = cam_dbg = win_dbg = {}
        motion_hint = "UNCERTAIN:0.0000"

        try:
            if not url:
                raise ValueError("Empty URL — skipping clip.")

            # ---- Download ----
            t0 = time.perf_counter()
            print("  Downloading...")
            local_path = download_video(url)
            t_download = time.perf_counter() - t0
            print(f"  Download: {t_download:.1f}s")

            # ---- Sample frames ----
            t0 = time.perf_counter()
            raw_frames, dur, picked_ts   = sample_timeline_frames(local_path, NUM_FRAMES)
            frames_full = [resize_image_keep_aspect(f, FRAME_SIZE_MAX) for f in raw_frames]

            motion_n = motion_frame_count_from_duration(dur)
            motion_raw, motion_dur, motion_ts = sample_timeline_frames(local_path, motion_n)
            motion_frames = [resize_image_keep_aspect(f, FRAME_SIZE_MAX) for f in motion_raw]

            t_frames = time.perf_counter() - t0

            frames_left:  List = []
            frames_right: List = []
            for fr in frames_full:
                l, r = split_left_right(fr)
                frames_left.append(l)
                frames_right.append(r)

            sel_dbg = {
                "vlm_picked_ts":    [round(x, 3) for x in picked_ts],
                "motion_picked_ts": [round(x, 3) for x in motion_ts],
                "dur_s":            round(float(dur), 3),
                "motion_dur_s":     round(float(motion_dur), 3),
                "vlm_frames":       len(frames_full),
                "motion_frames":    len(motion_frames),
            }
            drop_dbg = {"note": "not_used"}

            # ---- Save debug frames ----
            if SAVE_DEBUG_FRAMES:
                for idx, fr in enumerate(frames_left,  start=1):
                    fr.save(os.path.join(clip_d, f"frame_driver_{idx:02d}.jpg"),    quality=92)
                for idx, fr in enumerate(frames_right, start=1):
                    fr.save(os.path.join(clip_d, f"frame_passenger_{idx:02d}.jpg"), quality=92)

            # ---- Camera quality ----
            # Returns cam_s:
            #   0 = OK          → VLM runs normally
            #   2 = tier 1      → solid obstruction confirmed, skip VLM
            #   3 = blur        → skip VLM
            #   4 = foreground  → skip VLM, review flag
            #   5 = tier 2      → possible covering, VLM runs + review flag
            cam_s, cam_dbg = decide_camera_status_from_frames(frames_full)

            # ---- Motion ----
            vs, vs_label, vs_score, win_dbg = motion_hint_from_frames(motion_frames)
            motion_hint = f"{vs_label}:{vs_score:.4f}"

            # ---- Face obstruction 66 ----
            # Only check if camera is OK or tier 2 (VLM will run on these)
            if cam_s not in (2, 3, 4) and cam_o == 2:
                potential66, face66_dbg = detect_potential_face_obstruction_66(frames_left)
                if potential66:
                    rev_codes.append(EVENT_CODE["DRIVER_FACE_OBSTRUCTED"])

            # ---- VLM branch ----
            if cam_s == 4:
                # ── Foreground obstruction — skip VLM, add review flag ────────
                t_llm    = 0.0
                rev_codes = sorted(set(rev_codes + [EVENT_CODE["OBSTRUCTION_VIEW"]]))
                vlm_norm  = blank_vlm_result(reason="foreground_obstruction_review")
                print("  NOTE   : Foreground obstruction detected — VLM skipped")

            elif cam_s == 2:
                # ── Tier 1 — solid obstruction confirmed, skip VLM ────────────
                # High blob ratio + very low gradient confirmed by decide_camera_status_from_frames
                t_llm     = 0.0
                det_codes = sorted(set(det_codes + [EVENT_CODE["CAMERA_COVERED"]]))
                vlm_norm  = blank_vlm_result(reason="cam_covered_tier1")

                if event_description == "v_cam_covered":
                    labelled_checked = "y"
                    labelled_agreed  = labelled_agreement(
                        event_description=event_description,
                        labelled_checked=True,
                        cam_s=cam_s, seatbelt=seatbelt, phone=phone,
                        distraction=distraction, fatigue=fatigue, smoking=smoking, vs=vs,
                    )
                print("  NOTE   : Tier 1 — camera definitely covered, VLM skipped")

            elif cam_s == 3:
                # ── Blur — skip VLM ───────────────────────────────────────────
                t_llm    = 0.0
                vlm_norm = blank_vlm_result(reason="cam_blur")
                print("  NOTE   : Camera blur — VLM skipped")

            elif cam_s == 5:
                # ── Tier 2 — possible covering (cream/beads/dirty lens) ───────
                # Add camera covered review flag but still run VLM
                # VLM may detect driver behaviour through partial obstruction
                rev_codes = sorted(set(rev_codes + [EVENT_CODE["CAMERA_COVERED_REVIEW"]]))

                if event_description == "v_cam_covered":
                    labelled_checked = "y"
                    labelled_agreed  = "n"  # camera not fully confirmed covered

                print("  NOTE   : Tier 2 — possible covering, VLM will run with review flag")

                # Treat as cam_s=0 for VLM routing — fall through to VLM block below
                cam_s = 0

            # ── VLM runs for cam_s == 0 (including tier 2 reclassified above) ─
            if cam_s == 0:
                if event_description == "v_cam_covered" and cam_s == 0:
                    # Camera passed quality check but was labelled as covered
                    labelled_checked = "y"
                    labelled_agreed  = "n"

                if cam_o != 2:
                    # Not a cabin camera — skip VLM behaviour analysis
                    vlm_norm = blank_vlm_result(reason=f"input_camera_not_cabin(camera={camera_int})")

                else:
                    target = label_to_target(event_description)

                    if vs == 2:
                        # Vehicle STOPPED — only run smoking check (fast path)
                        if target == "smoking":
                            print("  Model: stopped + smoking label (targeted smoking)...")
                            labelled_checked = "y"
                        else:
                            print("  Model: stopped (smoking-only fast path)...")
                        t0 = time.perf_counter()
                        vlm_norm = run_targeted_check(frames_left, "smoking")
                        t_llm += time.perf_counter() - t0

                        if labelled_event == "y" and target not in (None, "cam_covered", "smoking"):
                            vlm_norm["why"] = (
                                vlm_norm.get("why", "") + " | stopped_non_smoke_label_not_checked"
                            ).strip()[:120]

                    elif labelled_event == "y" and target and target != "cam_covered":
                        # Labelled event — targeted check only
                        print(f"  Model: labelled targeted ({target})...")
                        t0 = time.perf_counter()
                        vlm_norm = run_targeted_check(frames_left, target)
                        t_llm += time.perf_counter() - t0
                        labelled_checked = "y"

                    else:
                        # Full driver-side analysis
                        print("  Model: full driver-side analysis...")
                        t0 = time.perf_counter()
                        raw_resp = ask_qwen_frames(
                            frames_left, full_prompt,
                            max_tokens=MAX_TOKENS, jpeg_quality=JPEG_QUALITY,
                        )
                        t_llm += time.perf_counter() - t0

                        raw_path = os.path.join(clip_d, "raw_response.txt")
                        with open(raw_path, "w", encoding="utf-8") as f:
                            f.write(raw_resp)

                        obj      = _extract_first_complete_json(raw_resp)
                        vlm_norm = normalize_vlm(obj, n_frames=NUM_FRAMES)

                    # Unpack VLM outputs
                    driver_vis  = int(vlm_norm["driver_vis"])
                    seatbelt    = str(vlm_norm["seatbelt"])
                    phone       = str(vlm_norm["phone"])
                    distraction = str(vlm_norm["distraction"])
                    fatigue     = str(vlm_norm.get("fatigue",  "u"))
                    smoking     = str(vlm_norm.get("smoking",  "u"))
                    eatdrink    = str(vlm_norm.get("eatdrink", "n"))
                    drinking    = str(vlm_norm.get("drinking", "u"))
                    alcohol     = str(vlm_norm.get("alcohol",  "u"))

                    # Cam-covered sanity check via VLM visibility
                    # If VLM can't see driver and vehicle isn't stopped,
                    # add obstruction review flag
                    if event_description == "v_cam_covered" and driver_vis != 1 and vs != 2:
                        rev_codes.append(EVENT_CODE["OBSTRUCTION_VIEW"])

                    # ---- Recheck seatbelt + phone if uncertain ----
                    if (RECHECK_ENABLED and cam_o == 2
                            and driver_vis == 1
                            and (seatbelt == "u" or phone == "u")):
                        try:
                            print("  Recheck: seatbelt + phone (cropped)...")
                            r2 = recheck_seatbelt_phone(frames_full, clip_d)
                            if seatbelt == "u" and r2.get("seatbelt") in ("y", "n"):
                                vlm_norm["seatbelt"]   = r2["seatbelt"]
                                vlm_norm["seatbelt_f"] = r2.get("seatbelt_f", [])
                            if phone == "u" and r2.get("phone") in ("y", "n"):
                                vlm_norm["phone"]   = r2["phone"]
                                vlm_norm["phone_f"] = r2.get("phone_f", [])
                            vlm_norm["why"]             = str(vlm_norm.get("why", "")).strip()[:120]
                            vlm_norm["why_recheck"]     = str(r2.get("why", "")).strip()[:120]
                            vlm_norm["recheck_seatbelt_phone"] = r2
                        except Exception as e:
                            vlm_norm["recheck_seatbelt_phone"] = {"error": str(e)[:160]}

                        # Re-read after recheck
                        seatbelt = str(vlm_norm["seatbelt"])
                        phone    = str(vlm_norm["phone"])

                    # ---- Decide events ----
                    det_tmp, rev_tmp = decide_events(
                        cam_s=cam_s, vs=vs, cam_o=cam_o,
                        driver_vis=driver_vis, seatbelt=seatbelt,
                        phone=phone, distraction=distraction,
                        fatigue=fatigue, smoking=smoking,
                    )
                    det_codes = sorted(set(det_codes + det_tmp))
                    rev_codes = sorted(set(rev_codes + rev_tmp))

                    # ---- Review scores ----
                    review_scores["seatbelt"] = _review_score_seatbelt(seatbelt, vs)
                    review_scores["phone"]    = _review_score_phone(phone, vs)
                    review_scores["drinking"] = _review_score_drinking(drinking, alcohol)
                    if review_scores["drinking"] == "high":
                        review_items.append("drinking_alcohol")

                    vlm_norm["review_scores"] = dict(review_scores)
                    if review_items:
                        vlm_norm["review_items"] = list(review_items)

                    # ---- Labelled agreement ----
                    labelled_agreed = labelled_agreement(
                        event_description=event_description,
                        labelled_checked=(labelled_checked == "y"),
                        cam_s=cam_s, seatbelt=seatbelt, phone=phone,
                        distraction=distraction, fatigue=fatigue,
                        smoking=smoking, vs=vs,
                    )

            # ---- Build output JSON ----
            t_total = time.perf_counter() - t0_total

            out: Dict[str, Any] = {
                "status": status,
                "timing": {
                    "total_s":    round(t_total,    6),
                    "download_s": round(t_download, 6),
                    "frames_s":   round(t_frames,   6),
                    "llm_s":      round(t_llm,      6),
                },
                "cam":   {"o": int(cam_o), "s": int(cam_s)},
                "scene": {"vs": int(vs)},
                "meta":  {
                    "event_description": event_description,
                    "labelled_event":    labelled_event,
                    "labelled_checked":  labelled_checked,
                    "labelled_agreed":   labelled_agreed,
                },
                "det": [{"e": int(c)} for c in sorted(set(det_codes))],
                "rev": [{"e": int(c)} for c in sorted(set(rev_codes))],
                "debug": {
                    "motion":        motion_hint,
                    "motion_detail": win_dbg,
                    "select":        sel_dbg,
                    "drop_bad":      drop_dbg,
                    "cam_quality":   cam_dbg,
                    "face66":        {"flag": int(potential66), "detail": face66_dbg},
                    "vlm":           vlm_norm,
                },
            }

            if OMIT_EMPTY_LISTS:
                for k in ("det", "rev"):
                    if not out.get(k):
                        out.pop(k, None)

            result_path = os.path.join(clip_d, RESULT_FILENAME)
            with open(result_path, "w", encoding="utf-8") as f:
                import json as _json
                _json.dump(out, f, ensure_ascii=False, indent=2)

            # ---- Save rerun progress ----------------------------------------
            # Appended after every successful clip so VM restarts don't lose work
            rerun_progress_path = os.path.join(OUTPUT_BASE_DIR, "rerun_progress.csv")
            write_header = not os.path.exists(rerun_progress_path)
            with open(rerun_progress_path, "a", newline="", encoding="utf-8") as pf:
                writer = csv.DictWriter(
                    pf, fieldnames=["clip_no","cam_s","tier","det_codes","rev_codes"]
                )
                if write_header:
                    writer.writeheader()
                tier_written = out.get("debug", {}).get("cam_quality", {}).get("tier", "")
                writer.writerow({
                    "clip_no"  : clip_no,
                    "cam_s"    : cam_s,
                    "tier"     : tier_written,
                    "det_codes": str([d["e"] for d in out.get("det", [])]),
                    "rev_codes": str([d["e"] for d in out.get("rev", [])]),
                })

            # ---- Console summary --------------------------------------------
            print(f"  Motion : {motion_hint}  stats={win_dbg.get('stats', {})}  first_last={win_dbg.get('first_last')}")
            print(f"  Cam    : o={cam_o} s={cam_s}  |  Scene: vs={vs}")
            print(f"  Label  : event={event_description}  checked={labelled_checked}  agreed={labelled_agreed}")
            if det_codes: print(f"  DET    : {sorted(set(det_codes))}")
            if rev_codes: print(f"  REV    : {sorted(set(rev_codes))}")
            if cam_s in (2, 3, 4):
                print("  NOTE   : Camera gated — behaviour VLM skipped")
            else:
                print(f"  VLM    : vis={vlm_norm.get('driver_vis')} sb={vlm_norm.get('seatbelt')} "
                      f"ph={vlm_norm.get('phone')} dst={vlm_norm.get('distraction')} "
                      f"fat={vlm_norm.get('fatigue')} smk={vlm_norm.get('smoking')} "
                      f"eat={vlm_norm.get('eatdrink')} | {vlm_norm.get('why', '')}")
            print(f"  Timing : total={t_total:.1f}s  dl={t_download:.1f}s  "
                  f"frames={t_frames:.1f}s  llm={t_llm:.1f}s")
            print(f"  Saved  : {os.path.abspath(result_path)}")

        except Exception as exc:
            err_msg = str(exc)
            print(f"  ERROR  : {err_msg}")
            with open(os.path.join(clip_d, ERROR_FILENAME), "w", encoding="utf-8") as f:
                f.write(err_msg)

            if "HTTP" in err_msg or "chat/completions" in err_msg:
                status = "FAILED_LLM"
            elif any(k in err_msg for k in ("Download attempt", "requests", "Connection")):
                status = "FAILED_DOWNLOAD"
            else:
                status = "FAILED_OTHER"

        finally:
            if local_path and os.path.exists(local_path):
                try: os.remove(local_path)
                except: pass

# -----------------------------------------------------------------------
# Write summary CSV from all results on disk
# -----------------------------------------------------------------------
summary_path = os.path.join(OUTPUT_BASE_DIR, "summary.csv")
write_summary_csv(
    all_video_urls_internal=ALL_VIDEO_URLS_INTERNAL,
    base=OUTPUT_BASE_DIR,
    path=summary_path,
)
print("=" * 80)
print("Done.")

In [ ]:
# =============================================================================
# TARGETED RERUN — OLD TIER 1 COVERED CLIPS
# =============================================================================
# Finds all clips where result.json has det_codes containing CAMERA_COVERED (6)
# and cam_quality tier is not set (old pipeline).
# Deletes their result.json so they re-enter PENDING_JOBS.
# Saves progress to rerun_progress.csv after each clip so VM restarts are safe.
# =============================================================================

import json
import os
import re
import csv
from pathlib import Path

RERUN_PROGRESS_PATH = os.path.join(OUTPUT_BASE_DIR, "rerun_progress.csv")
CAMERA_COVERED_CODE = EVENT_CODE["CAMERA_COVERED"]  # 6

# ── Load already completed reruns ─────────────────────────────────────────────
already_done = set()
if os.path.exists(RERUN_PROGRESS_PATH):
    with open(RERUN_PROGRESS_PATH, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            already_done.add(int(row["clip_no"]))
print(f"Already rerun from previous session: {len(already_done)} clips")

# ── Find clips to rerun ───────────────────────────────────────────────────────
to_rerun = []
for name in sorted(os.listdir(OUTPUT_BASE_DIR)):
    if not re.fullmatch(r"clip_\d+", name):
        continue
    clip_no     = int(name.split("_")[1])
    if clip_no in already_done:
        continue
    result_path = os.path.join(OUTPUT_BASE_DIR, name, RESULT_FILENAME)
    if not os.path.exists(result_path):
        continue
    try:
        with open(result_path) as f:
            result = json.load(f)
        det_codes = [d["e"] for d in result.get("det", []) if "e" in d]
        cam_s     = result.get("cam", {}).get("s", 0)
        tier      = result.get("debug", {}).get("cam_quality", {}).get("tier", None)

        # Only rerun old tier 1 covered clips — those with det=6, cam_s=2,
        # and no tier key (written by old pipeline)
        if CAMERA_COVERED_CODE in det_codes and cam_s == 2 and tier is None:
            to_rerun.append(clip_no)
    except Exception:
        continue

print(f"Clips to rerun                      : {len(to_rerun)}")
print(f"First 20: {sorted(to_rerun)[:20]}")

# ── Delete result.json for these clips so they re-enter PENDING_JOBS ──────────
if len(to_rerun) > 0:
    confirm = input(f"\nDelete {len(to_rerun)} result.json files to trigger rerun? (yes/no): ")
    if confirm.strip().lower() == "yes":
        for clip_no in to_rerun:
            result_path = os.path.join(OUTPUT_BASE_DIR, f"clip_{clip_no:03d}", RESULT_FILENAME)
            if os.path.exists(result_path):
                os.remove(result_path)
                print(f"  Deleted: clip_{clip_no:03d}/result.json")
        print(f"\n✓ Deleted {len(to_rerun)} result.json files")
        print(f"Now re-run Cells 6 and 8 to reprocess these clips.")
        print(f"Progress will be saved to: {RERUN_PROGRESS_PATH}")
    else:
        print("Cancelled.")
else:
    print("Nothing to rerun.")

## Repeatability: Rerun 5% of videos and compare to original results

In [ ]:
"""
CELL 9 — REPEATABILITY RERUN
==============================
Re-runs a sample of clips through the EXACT same code path as Cell 8.

The routing decision (which prompt to use) is derived identically:
  - vs==2 (stopped)            → smoking-only  (same as Cell 8)
  - labelled event + target    → targeted check (same as Cell 8)
  - everything else            → full prompt    (same as Cell 8)

vs and event_description are read from the original result.json so the
rerun makes the same routing decision Cell 8 made.

Frames are loaded from disk (no re-download).
Recheck also runs if needed — identical to Cell 8.

Output:
  clip_NNN/result_rerun.json   — rerun VLM output
  debug_frames_qwen/rerun_comparison.csv — field-by-field comparison

Mismatch definition:
  TRUE mismatch  = y→n or n→y  (definitive flip, counts as failure)
  Certainty change = u→y, y→u, u→n, n→u  (not a failure, just confidence)
  Match          = same value

Depends on: Cells 1–6 must be run first.
"""

import json
import os
import random
import time
from pathlib import Path
from typing  import Any, Dict, List

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

RERUN_SAMPLE_N        = 240      
RERUN_FINAL_PERCENT   = 0.05    # 5% threshold warning
RERUN_RANDOM_SEED     = 42      # fixed seed = same clips every time
RERUN_RESULT_FILENAME = "result_rerun.json"
RERUN_COMPARISON_CSV  = os.path.join(OUTPUT_BASE_DIR, "rerun_comparison.csv")
COMPARE_FIELDS        = ["seatbelt", "phone", "distraction", "fatigue", "smoking", "eatdrink"]

# ─────────────────────────────────────────────────────────────────────────────
# SAFETY GUARD
# ─────────────────────────────────────────────────────────────────────────────
for _req in ("OUTPUT_BASE_DIR", "ask_qwen_frames", "normalize_vlm",
             "run_targeted_check", "build_prompt", "label_to_target",
             "blank_vlm_result", "_extract_first_complete_json",
             "recheck_seatbelt_phone", "LABELLED_EVENTS", "RESULT_FILENAME",
             "NUM_FRAMES", "MAX_TOKENS", "JPEG_QUALITY", "RECHECK_ENABLED"):
    if _req not in dir():
        raise NameError(f"'{_req}' not defined — run Cells 1–6 first.")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _load_driver_frames_from_disk(clip_dir: str) -> List:
    """
    Load frame_driver_*.jpg files as PIL Images in frame order.
    These are the left-half driver frames — exactly what the VLM saw originally.
    """
    from PIL import Image as _PIL
    frames = []
    for p in sorted(Path(clip_dir).glob("frame_driver_*.jpg")):
        try:
            frames.append(_PIL.open(str(p)).convert("RGB"))
        except Exception as e:
            print(f"    WARNING: could not load {p.name}: {e}")
    return frames


def _load_full_frames_from_disk(clip_dir: str) -> List:
    """
    Reconstruct full-width frames by stitching driver + passenger halves.
    Required for recheck_seatbelt_phone which crops from the full frame.
    """
    from PIL import Image as _PIL
    driver_paths    = sorted(Path(clip_dir).glob("frame_driver_*.jpg"))
    passenger_paths = sorted(Path(clip_dir).glob("frame_passenger_*.jpg"))
    full_frames = []
    for dp, pp in zip(driver_paths, passenger_paths):
        try:
            left  = _PIL.open(str(dp)).convert("RGB")
            right = _PIL.open(str(pp)).convert("RGB")
            full  = _PIL.new("RGB", (left.width + right.width,
                                     max(left.height, right.height)))
            full.paste(left,  (0, 0))
            full.paste(right, (left.width, 0))
            full_frames.append(full)
        except Exception as e:
            print(f"    WARNING: could not reconstruct frame {dp.name}: {e}")
    return full_frames


def _read_original_result(clip_dir: str) -> Dict[str, Any]:
    """Read the full original result.json."""
    try:
        with open(os.path.join(clip_dir, RESULT_FILENAME), "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


def _compare_field(orig: str, rerun: str) -> str:
    """
    match          — identical
    mismatch       — y→n or n→y  ← only this is a repeatability failure
    hardened       — u→y or u→n  (model more decisive, not a problem)
    softened       — y→u or n→u  (model less decisive, not a problem)
    both_uncertain — u→u
    """
    if orig == rerun:
        return "match"
    if orig == "u" and rerun == "u":
        return "both_uncertain"
    if orig == "u":
        return "hardened"
    if rerun == "u":
        return "softened"
    return "mismatch"


def _discover_processable_clips() -> tuple[List[Dict], List[Dict]]:
    """
    Returns (already_rerun, not_yet_rerun).
    already_rerun = clips with a valid result_rerun.json on disk.
    not_yet_rerun = clips that passed all filters but haven't been rerun yet.
    """
    import re as _re
    already_rerun = []
    not_yet_rerun = []

    if not os.path.isdir(OUTPUT_BASE_DIR):
        return already_rerun, not_yet_rerun

    for name in sorted(os.listdir(OUTPUT_BASE_DIR)):
        m = _re.fullmatch(r"clip_(\d+)", name)
        if not m:
            continue
        clip_no  = int(m.group(1))
        clip_dir = os.path.join(OUTPUT_BASE_DIR, name)
        result   = _read_original_result(clip_dir)
        if not result:
            continue
        if int(result.get("cam", {}).get("s", 0)) in (2, 3, 4):
            continue
        if str(result.get("status", "OK")).upper() not in ("OK", "SUCCESS", ""):
            continue
        if not list(Path(clip_dir).glob("frame_driver_*.jpg")):
            continue

        meta              = result.get("meta", {})
        event_description = str(meta.get("event_description", ""))
        vs                = int(result.get("scene", {}).get("vs", 0))
        det_codes         = [d["e"] for d in result.get("det", []) if "e" in d]
        rev_codes         = [d["e"] for d in result.get("rev", []) if "e" in d]
        all_codes         = det_codes + rev_codes
        label_map         = {5: "seatbelt", 10: "phone", 11: "distraction",
                             13: "fatigue", 15: "smoking", 4: "obstruction", 66: "face_blocked"}
        primary = next(
            (label_map[c] for c in (10, 5, 15, 13, 11, 66, 4) if c in all_codes),
            "no_event"
        )

        clip_entry = {
            "clip_no":           clip_no,
            "clip_dir":          clip_dir,
            "event_description": event_description,
            "vs":                vs,
            "primary_det":       primary,
        }

        # Check if already successfully rerun
        rerun_path = os.path.join(clip_dir, RERUN_RESULT_FILENAME)
        if os.path.exists(rerun_path):
            try:
                with open(rerun_path) as f:
                    existing = json.load(f)
                if existing.get("rerun_status") == "ok":
                    already_rerun.append(clip_entry)
                    continue
            except Exception:
                pass

        not_yet_rerun.append(clip_entry)

    return already_rerun, not_yet_rerun


def _stratified_sample(clips: List[Dict], n: int, seed: int) -> List[Dict]:
    """
    Proportional stratified sample across primary_det categories.
    Fixed seed = same clips every single time — essential for reproducibility.
    """
    from collections import defaultdict
    random.seed(seed)
    groups: Dict[str, list] = defaultdict(list)
    for clip in clips:
        groups[clip["primary_det"]].append(clip)

    sample  = []
    per_cat = max(1, n // len(groups)) if groups else n
    for cat_clips in groups.values():
        sample.extend(random.sample(cat_clips, min(per_cat, len(cat_clips))))

    sampled_nos = {c["clip_no"] for c in sample}
    remaining   = [c for c in clips if c["clip_no"] not in sampled_nos]
    shortfall   = n - len(sample)
    if shortfall > 0 and remaining:
        sample.extend(random.sample(remaining, min(shortfall, len(remaining))))

    return sample[:n]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — DISCOVER & SAMPLE
# ─────────────────────────────────────────────────────────────────────────────

already_rerun, not_yet_rerun = _discover_processable_clips()
total = len(already_rerun) + len(not_yet_rerun)
five_pct = max(1, int(total * RERUN_FINAL_PERCENT))

print("=" * 70)
print("REPEATABILITY RERUN")
print("=" * 70)
print(f"  VLM-analysed clips available : {total}")
print(f"  Already rerun on disk        : {len(already_rerun)}")
print(f"  5% threshold                 : {five_pct}")
print(f"  RERUN_SAMPLE_N (this run)    : {RERUN_SAMPLE_N}")

if total >= 500 and RERUN_SAMPLE_N < five_pct:
    print(f"\n  ⚠  WARNING: set RERUN_SAMPLE_N >= {five_pct} for thesis final run.")
else:
    print(f"\n  ✓  Sample size OK.")

# Use already-rerun clips first, top up with new sample if needed
if len(already_rerun) >= RERUN_SAMPLE_N:
    sample_clips = already_rerun[:RERUN_SAMPLE_N]
    print(f"\n  Quota met by existing reruns — no new clips will be processed.")
else:
    shortfall    = RERUN_SAMPLE_N - len(already_rerun)
    new_sample   = _stratified_sample(not_yet_rerun, shortfall, RERUN_RANDOM_SEED)
    sample_clips = already_rerun + new_sample
    print(f"\n  Using {len(already_rerun)} existing + {len(new_sample)} new clips.")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — RERUN (IDENTICAL ROUTING TO CELL 8)
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'─'*70}")
print("Running VLM rerun...")
print(f"{'─'*70}\n")

full_prompt = build_prompt()
rerun_rows: List[Dict[str, Any]] = []

for run_idx, clip_info in enumerate(sample_clips, start=1):
    clip_no           = clip_info["clip_no"]
    clip_dir          = clip_info["clip_dir"]
    event_description = clip_info["event_description"]
    vs                = clip_info["vs"]          # from original run
    out_path          = os.path.join(clip_dir, RERUN_RESULT_FILENAME)

    # Skip if already successfully rerun
    if os.path.exists(out_path):
        try:
            with open(out_path) as f:
                existing = json.load(f)
            if existing.get("rerun_status") == "ok":
                #print(f"[{run_idx}/{len(sample_clips)}] clip_{clip_no:03d} — already done")
                # Reconstruct the flat row structure from saved comparison dict
                row = {
                    "rerun_status":      "ok",
                    "clip_no":           clip_no,
                    "event_description": clip_info["event_description"],
                    "vs":                clip_info["vs"],
                    "primary_det":       clip_info["primary_det"],
                    "prompt_used":       existing.get("prompt_used", ""),
                    "t_llm_s":           existing.get("t_llm_s", 0),
                    "true_mismatch":     existing.get("true_mismatch", False),
                    "any_change":        existing.get("any_change", False),
                }
                orig_vlm  = _read_original_result(clip_dir).get("debug", {}).get("vlm", {})
                rerun_vlm = existing.get("vlm_rerun", {})
                comparison = existing.get("comparison", {})
                for field in COMPARE_FIELDS:
                    row[f"orig_{field}"]  = str(orig_vlm.get(field, "u"))
                    row[f"rerun_{field}"] = str(rerun_vlm.get(field, "u"))
                    row[f"cmp_{field}"]   = comparison.get(field, "match")
                rerun_rows.append(row)
                continue
        except Exception:
            pass

    print(f"[{run_idx}/{len(sample_clips)}] clip_{clip_no:03d}  "
          f"event={event_description!r}  vs={vs}  primary={clip_info['primary_det']}")

    frames_left = _load_driver_frames_from_disk(clip_dir)
    if len(frames_left) < 2:
        print(f"  ERROR: only {len(frames_left)} frames on disk")
        err = {"rerun_status": "error", "clip_no": clip_no,
               "error": f"only_{len(frames_left)}_frames"}
        with open(out_path, "w") as f:
            json.dump(err, f, indent=2)
        rerun_rows.append(err)
        continue

    t0 = time.perf_counter()
    try:
        # ─────────────────────────────────────────────────────────────────
        # EXACT SAME ROUTING LOGIC AS CELL 8
        # Copy-pasted and adapted for disk-loaded frames.
        # ─────────────────────────────────────────────────────────────────
        labelled_event = "y" if event_description in LABELLED_EVENTS else "n"
        target         = label_to_target(event_description)

        if vs == 2:
            # STOPPED — smoking-only fast path (same as Cell 8)
            if target == "smoking":
                print("  → stopped + smoking label (targeted smoking)...")
                labelled_checked_rerun = "y"
            else:
                print("  → stopped (smoking-only fast path)...")
                labelled_checked_rerun = "n"
            vlm_rerun  = run_targeted_check(frames_left, "smoking")
            prompt_key = "targeted_smoking"

        elif labelled_event == "y" and target and target != "cam_covered":
            # LABELLED EVENT — targeted check (same as Cell 8)
            print(f"  → targeted ({target})...")
            vlm_rerun  = run_targeted_check(frames_left, target)
            prompt_key = f"targeted_{target}"

        else:
            # FULL analysis (same as Cell 8)
            print(f"  → full analysis...")
            raw_resp   = ask_qwen_frames(
                frames_left, full_prompt,
                max_tokens=MAX_TOKENS, jpeg_quality=JPEG_QUALITY,
            )
            obj        = _extract_first_complete_json(raw_resp)
            vlm_rerun  = normalize_vlm(obj, n_frames=NUM_FRAMES)
            prompt_key = "full"

        # ── Recheck seatbelt/phone — same as Cell 8 ──────────────────────
        driver_vis = int(vlm_rerun.get("driver_vis", 0))
        seatbelt   = str(vlm_rerun.get("seatbelt", "u"))
        phone      = str(vlm_rerun.get("phone",    "u"))

        if (RECHECK_ENABLED and driver_vis == 1
                and (seatbelt == "u" or phone == "u")):
            try:
                print(f"  → recheck...")
                frames_full_rerun = _load_full_frames_from_disk(clip_dir)
                if frames_full_rerun:
                    r2 = recheck_seatbelt_phone(frames_full_rerun, clip_dir)
                    if seatbelt == "u" and r2.get("seatbelt") in ("y", "n"):
                        vlm_rerun["seatbelt"]   = r2["seatbelt"]
                        vlm_rerun["seatbelt_f"] = r2.get("seatbelt_f", [])
                    if phone == "u" and r2.get("phone") in ("y", "n"):
                        vlm_rerun["phone"]   = r2["phone"]
                        vlm_rerun["phone_f"] = r2.get("phone_f", [])
            except Exception as e:
                print(f"    recheck error (non-fatal): {e}")

        t_llm = time.perf_counter() - t0

        # ── Compare to original ───────────────────────────────────────────
        orig_vlm = _read_original_result(clip_dir).get("debug", {}).get("vlm", {})

        row: Dict[str, Any] = {
            "rerun_status":      "ok",
            "clip_no":           clip_no,
            "event_description": event_description,
            "vs":                vs,
            "primary_det":       clip_info["primary_det"],
            "prompt_used":       prompt_key,
            "t_llm_s":           round(t_llm, 2),
        }

        true_mismatch = False
        any_change    = False

        for field in COMPARE_FIELDS:
            orig_val  = str(orig_vlm.get(field, "u"))
            rerun_val = str(vlm_rerun.get(field, "u"))
            row[f"orig_{field}"]  = orig_val
            row[f"rerun_{field}"] = rerun_val
            cmp = _compare_field(orig_val, rerun_val)
            row[f"cmp_{field}"] = cmp
            if cmp == "mismatch":
                true_mismatch = True
                any_change    = True
            elif cmp in ("hardened", "softened"):
                any_change = True

        row["true_mismatch"] = true_mismatch
        row["any_change"]    = any_change

        # Save to disk
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump({
                "rerun_status":  "ok",
                "clip_no":       clip_no,
                "prompt_used":   prompt_key,
                "t_llm_s":       round(t_llm, 2),
                "vlm_rerun":     {k: vlm_rerun.get(k)
                                  for k in COMPARE_FIELDS + ["driver_vis", "why"]},
                "comparison":    {f: row[f"cmp_{f}"] for f in COMPARE_FIELDS},
                "true_mismatch": true_mismatch,
                "any_change":    any_change,
            }, f, ensure_ascii=False, indent=2)

        rerun_rows.append(row)

        # Console
        icon = "✓" if not true_mismatch else "✗"
        print(f"  {icon}  orig : sb={orig_vlm.get('seatbelt','u')} "
              f"ph={orig_vlm.get('phone','u')} "
              f"dst={orig_vlm.get('distraction','u')} "
              f"fat={orig_vlm.get('fatigue','u')} "
              f"smk={orig_vlm.get('smoking','u')}")
        print(f"     rerun: sb={vlm_rerun.get('seatbelt','u')} "
              f"ph={vlm_rerun.get('phone','u')} "
              f"dst={vlm_rerun.get('distraction','u')} "
              f"fat={vlm_rerun.get('fatigue','u')} "
              f"smk={vlm_rerun.get('smoking','u')}")

        mismatches = [f for f in COMPARE_FIELDS if row.get(f"cmp_{f}") == "mismatch"]
        changes    = [f for f in COMPARE_FIELDS
                      if row.get(f"cmp_{f}") in ("hardened", "softened")]
        if mismatches:
            print(f"     ✗ TRUE MISMATCH (y↔n): {mismatches}")
        if changes and not mismatches:
            print(f"     ~ certainty change only (not a failure): {changes}")

    except Exception as exc:
        print(f"  ERROR: {exc}")
        err = {"rerun_status": "error", "clip_no": clip_no, "error": str(exc)[:200]}
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(err, f, indent=2)
        rerun_rows.append(err)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — REPORT
# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — REPORT
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ok_rows  = [r for r in rerun_rows if r.get("rerun_status") == "ok"]

if ok_rows:
    rep_df   = pd.DataFrame(ok_rows)
    total_ok = len(ok_rows)
    true_mis = int(rep_df["true_mismatch"].sum())
    changes  = int(rep_df["any_change"].sum())

    # Count hardened / softened across all fields
    hardened_total = sum(
        (rep_df[f"cmp_{f}"] == "hardened").sum() for f in COMPARE_FIELDS
        if f"cmp_{f}" in rep_df.columns
    )
    softened_total = sum(
        (rep_df[f"cmp_{f}"] == "softened").sum() for f in COMPARE_FIELDS
        if f"cmp_{f}" in rep_df.columns
    )

    print(f"\n{'='*70}")
    print(f"REPEATABILITY REPORT  —  {total_ok} clips")
    print(f"{'='*70}")
    print(f"\n  No change at all               : "
          f"{total_ok - changes}/{total_ok} ({100*(total_ok-changes)/total_ok:.1f}%)")
    print(f"  Certainty change only (u↔y/n)  : "
          f"{changes - true_mis}/{total_ok} ({100*(changes-true_mis)/total_ok:.1f}%)")
    print(f"    of which hardened (u→y/n)    : {hardened_total} field-level changes")
    print(f"    of which softened (y/n→u)    : {softened_total} field-level changes")
    print(f"  TRUE mismatch (y↔n flip)       : "
          f"{true_mis}/{total_ok} ({100*true_mis/total_ok:.1f}%)")

    print(f"\n  Per-field breakdown:")
    print(f"  {'Field':<14} {'Match%':>7}  {'Mismatch':>9}  "
          f"{'Hardened':>9}  {'Softened':>9}")
    print(f"  {'─'*54}")
    for field in COMPARE_FIELDS:
        col = f"cmp_{field}"
        if col not in rep_df.columns:
            continue
        n          = total_ok
        match_n    = (rep_df[col] == "match").sum()
        mismatch_n = (rep_df[col] == "mismatch").sum()
        hard_n     = (rep_df[col] == "hardened").sum()
        soft_n     = (rep_df[col] == "softened").sum()
        print(f"  {field:<14} {100*match_n/n:>6.1f}%  "
              f"{mismatch_n:>9}  {hard_n:>9}  {soft_n:>9}")

    print(f"\n  By event category:")
    if "primary_det" in rep_df.columns:
        display(
            rep_df.groupby("primary_det")
            .agg(clips=("true_mismatch","count"),
                 true_mismatches=("true_mismatch","sum"),
                 certainty_changes=("any_change","sum"))
            .assign(mismatch_pct=lambda x:
                    (100*x["true_mismatches"]/x["clips"]).round(1))
            .reset_index()
        )

    # ── GRAPH 1 — Overall repeatability outcome breakdown ─────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Left: pie of overall outcomes
    ax = axes[0]
    no_change_n  = total_ok - changes
    cert_change_n = changes - true_mis
    labels = ["No change", "Certainty change\n(u↔y/n)", "True mismatch\n(y↔n)"]
    sizes  = [no_change_n, cert_change_n, true_mis]
    colors = ["#4CAF50", "#FF9800", "#F44336"]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors,
        autopct="%1.1f%%", startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5}
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax.set_title("Overall Repeatability\nOutcome Distribution", fontsize=11)

    # Middle: per-field match rates bar chart
    ax = axes[1]
    field_match_pcts = []
    for field in COMPARE_FIELDS:
        col = f"cmp_{field}"
        if col in rep_df.columns:
            field_match_pcts.append(100 * (rep_df[col] == "match").sum() / total_ok)
        else:
            field_match_pcts.append(0)

    bars = ax.barh(COMPARE_FIELDS, field_match_pcts, color="#2196F3", edgecolor="white")
    ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=9)
    ax.set_xlim(0, 110)
    ax.axvline(90, color="red", linewidth=1, linestyle="--", alpha=0.6, label="90% line")
    ax.legend(fontsize=8)
    ax.set_title("Per-Field Match Rate\n(% identical across reruns)", fontsize=11)
    ax.set_xlabel("Match %")

    # Right: hardened vs softened stacked bar per field
    ax = axes[2]
    hard_vals = []
    soft_vals = []
    for field in COMPARE_FIELDS:
        col = f"cmp_{field}"
        if col in rep_df.columns:
            hard_vals.append((rep_df[col] == "hardened").sum())
            soft_vals.append((rep_df[col] == "softened").sum())
        else:
            hard_vals.append(0)
            soft_vals.append(0)

    y = range(len(COMPARE_FIELDS))
    ax.barh(list(y), hard_vals, color="#FF9800", edgecolor="white", label="Hardened (u→y/n)")
    ax.barh(list(y), soft_vals, left=hard_vals, color="#9C27B0", edgecolor="white", label="Softened (y/n→u)")
    ax.set_yticks(list(y))
    ax.set_yticklabels(COMPARE_FIELDS)
    ax.set_title("Certainty Changes per Field\n(not failures — confidence shifts only)", fontsize=11)
    ax.set_xlabel("Number of clips")
    ax.legend(fontsize=8)

    plt.suptitle(
        f"Repeatability Analysis — {total_ok} clips  |  "
        f"Seed={RERUN_RANDOM_SEED}  |  True mismatch rate: {100*true_mis/total_ok:.1f}%",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    plt.show()

    # ── GRAPH 2 — Mismatch rate by event category ─────────────────────────
    if "primary_det" in rep_df.columns:
        cat_df = (
            rep_df.groupby("primary_det")
            .agg(clips=("true_mismatch", "count"),
                 true_mismatches=("true_mismatch", "sum"),
                 certainty_changes=("any_change", "sum"))
            .assign(mismatch_pct=lambda x: (100 * x["true_mismatches"] / x["clips"]).round(1))
            .reset_index()
            .sort_values("mismatch_pct", ascending=True)
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.barh(cat_df["primary_det"], cat_df["mismatch_pct"],
                       color="#F44336", edgecolor="white", alpha=0.85)
        ax.bar_label(
            bars,
            labels=[f"{p}%  (n={n})" for p, n in
                    zip(cat_df["mismatch_pct"], cat_df["clips"])],
            padding=4, fontsize=9
        )
        ax.axvline(5, color="black", linewidth=1, linestyle="--",
                   alpha=0.5, label="5% threshold")
        ax.legend(fontsize=8)
        ax.set_title("True Mismatch Rate by Event Category\n(y↔n flips only)", fontsize=11)
        ax.set_xlabel("Mismatch %")
        plt.tight_layout()
        plt.show()

    save_cols = (
        ["clip_no", "event_description", "vs", "primary_det", "prompt_used",
         "true_mismatch", "any_change"]
        + [f"orig_{f}"  for f in COMPARE_FIELDS]
        + [f"rerun_{f}" for f in COMPARE_FIELDS]
        + [f"cmp_{f}"   for f in COMPARE_FIELDS]
        + ["t_llm_s"]
    )
    rep_df[[c for c in save_cols if c in rep_df.columns]].to_csv(
        RERUN_COMPARISON_CSV, index=False, encoding="utf-8"
    )
    print(f"\n  Saved: {os.path.abspath(RERUN_COMPARISON_CSV)}")

else:
    print("No successful reruns to report.")

# csv of results

In [ ]:
import pandas as pd
from IPython.display import display

sumeri_df = pd.read_csv("debug_frames_qwen/summary.csv")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

display(sumeri_df)

In [ ]:
sumeri_df[["t_total_s", "t_download_s", "t_frames_s", "t_llm_s"]].agg(["mean", "count"]).round(2)